# Step Analysis Анлиз на стъпките в периода 2017-2026

In [ ]:
using Pkg

# 1. Temporarily disable automatic precompilation to prevent Jupyter timeouts
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"

# 2. Define all your packages in a single list
packages = [
    "CSV", "DataFrames", "Dates", "Statistics", "LinearAlgebra",
    "Plots", "StatsPlots", "Distributions", "Printf", "Random", 
    "StatsBase", "ARCHModels", "XLSX", "GLM", "HypothesisTests", 
    "MultivariateStats", "Measures", "Colors", 
]

# 3. Install them all in one efficient bulk action
Pkg.add(packages)

# Processing Excel Data with Julia and XLSX  
## Обработка на Excel данни с Julia и XLSX  

This notebook demonstrates how to load multiple sheets from an Excel file, clean and standardize them, and combine them into a single DataFrame using Julia.  
Този бележник демонстрира как да заредите множество листове от Excel файл, да ги почистите и стандартизирате, и да ги комбинирате в един DataFrame с помощта на Julia.

---

### 1. Install required packages (if not already installed)  
### 1. Инсталирайте необходимите пакети (ако все още не са инсталирани)  

```julia
using Pkg
Pkg.add(["XLSX", "DataFrames"])
```

---

### 2. Load the packages and read the Excel file  
### 2. Заредете пакетите и прочетете Excel файла  

```julia
using XLSX, DataFrames

# Path to your Excel file – adjust as needed
# Път до вашия Excel файл – коригирайте според нуждите
file_path = raw"D:\data\projects\steps\Steps.xlsx"
xf = XLSX.readxlsx(file_path)
```

---

### 3. Get sheet names (skip "Analysis") and process each year sheet  
### 3. Вземете имената на листовете (пропуснете "Analysis") и обработете всеки годишен лист  

We filter out any sheet named "Analysis". For each remaining sheet we:  
- Read the table into a temporary DataFrame.  
- Fix a common typo: rename "Colories" to "Calories" if present.  
- Add a `Number` column (1,2,3,…) if it does not already exist (needed for sheets 2025, 2026).  
- Ensure all columns exist (add missing ones with `missing` values).  
- Reorder columns to a consistent order.  
- Store each processed DataFrame in a list.  

Изключваме всички листове с име "Analysis". За всеки останал лист:  
- Четем таблицата във временен DataFrame.  
- Поправяме често срещана грешка: преименуваме "Colories" на "Calories", ако съществува.  
- Добавяме колона `Number` (1,2,3,…), ако не съществува (необходимо за листове 2025, 2026).  
- Гарантираме, че всички колони съществуват (добавяме липсващите със стойности `missing`).  
- Подреждаме колоните в последователен ред.  
- Запазваме всеки обработен DataFrame в списък.  

```julia
year_sheets = filter(s -> s != "Analysis", XLSX.sheetnames(xf))
all_data = DataFrame[]

for sheet in year_sheets
    df_temp = DataFrame(XLSX.readtable(file_path, sheet))
    
    # Fix typo: rename "Colories" to "Calories" if it exists
    # Поправка: преименуваме "Colories" на "Calories", ако съществува
    if "Colories" in names(df_temp)
        rename!(df_temp, "Colories" => "Calories")
    end
    
    # Add Number column if missing (2025, 2026)
    # Добавяме колона Number, ако липсва (2025, 2026)
    if !("Number" in names(df_temp))
        insertcols!(df_temp, 1, :Number => 1:nrow(df_temp))
    end
    
    # Ensure all DataFrames have the same column order
    # Гарантираме, че всички DataFrames имат еднакъв ред на колоните
    full_cols = ["Number", "Day", "Steps", "Difference", "Calories", 
                 "Difference of C", "Distance", "Difference of D", 
                 "Time", "Difference of time"]
    
    # Add missing columns with missing values
    # Добавяме липсващите колони със стойности missing
    for col in full_cols
        if !(col in names(df_temp))
            df_temp[!, col] = [missing for _ in 1:nrow(df_temp)]
        end
    end
    
    # Reorder columns
    # Пренареждаме колоните
    df_temp = df_temp[:, full_cols]
    
    push!(all_data, df_temp)
    println("Processed: $sheet => ", nrow(df_temp), " rows")
end
```

---

### 4. Combine all sheets into one DataFrame  
### 4. Комбинирайте всички листове в един DataFrame  

```julia
df_combined = vcat(all_data...)

println("\n=== Combined DataFrame ===")
println("Total rows: ", nrow(df_combined))
println("Columns: ", names(df_combined))
```

---

### 5. Preview the data and check for missing values  
### 5. Преглед на данните и проверка за липсващи стойности  

```julia
println("\nFirst 5 rows:")
first(df_combined, 5)

println("\nLast 5 rows:")
last(df_combined, 5)

println("\nMissing values per column:")
for col in names(df_combined)
    n_missing = sum(ismissing.(df_combined[!, col]))
    println("  $col: $n_missing")
end
```

---

### Expected output  
### Очакван резултат  

The code will print:  
- For each sheet: the number of rows processed.  
- A summary of the combined DataFrame: total rows and column names.  
- The first 5 and last 5 rows.  
- The count of missing values in each column.  

Кодът ще изведе:  
- За всеки лист: броя на обработените редове.  
- Обобщение на комбинирания DataFrame: общ брой редове и имена на колоните.  
- Първите 5 и последните 5 реда.  
- Броя на липсващите стойности във всяка колона.  

---

### Notes / Бележки  

- Adjust the `file_path` variable to point to your actual Excel file.  
- The code assumes that all sheets (except "Analysis") have the same column structure after standardization; missing columns are filled with `missing`.  
- The `Number` column is added to sheets that lack it, to maintain a row index.  

- Променете променливата `file_path`, за да сочи към вашия действителен Excel файл.  
- Кодът предполага, че всички листове (освен "Analysis") имат еднаква структура на колоните след стандартизация; липсващите колони се попълват с `missing`.  
- Колоната `Number` се добавя към листове, които я нямат, за да се поддържа индекс на редовете.  

---

**Happy coding!**  
**Приятно програмиране!**

In [ ]:
using XLSX
using DataFrames

file_path = raw"D:\data\projects\steps\Steps.xlsx"
xf = XLSX.readxlsx(file_path)

year_sheets = filter(s -> s != "Analysis", XLSX.sheetnames(xf))

all_data = DataFrame[]

for sheet in year_sheets
    df_temp = DataFrame(XLSX.readtable(file_path, sheet))
    
    # Fix typo: rename "Colories" to "Calories" if it exists
    if "Colories" in names(df_temp)
        rename!(df_temp, "Colories" => "Calories")
    end
    
    # Add Number column if missing (2025, 2026)
    if !("Number" in names(df_temp))
        insertcols!(df_temp, 1, :Number => 1:nrow(df_temp))
    end
    
    # Ensure all DataFrames have the same column order
    # Define the full column set
    full_cols = ["Number", "Day", "Steps", "Difference", "Calories", 
                 "Difference of C", "Distance", "Difference of D", 
                 "Time", "Difference of time"]
    
    # Add missing columns with missing values
    for col in full_cols
        if !(col in names(df_temp))
            df_temp[!, col] = [missing for _ in 1:nrow(df_temp)]
        end
    end
    
    # Reorder columns
    df_temp = df_temp[:, full_cols]
    
    push!(all_data, df_temp)
    println("Processed: $sheet => ", nrow(df_temp), " rows")
end

# Combine all
df_combined = vcat(all_data...)

println("\n=== Combined DataFrame ===")
println("Total rows: ", nrow(df_combined))
println("Columns: ", names(df_combined))
println("\nFirst 5 rows:")
first(df_combined, 5)

println("\nLast 5 rows:")
last(df_combined, 5)

# Check for missing values per column
println("\nMissing values per column:")
for col in names(df_combined)
    n_missing = sum(ismissing.(df_combined[!, col]))
    println("  $col: $n_missing")
end

# Interpretation of the Processing Results  
## Интерпретация на резултатите от обработката  

After running the code, we obtained the following output. Below we analyse what these numbers mean and what we can learn from them.

---

### 1. Data Coverage by Year  
### 1. Покритие на данните по години  

| Year | Rows | Comment |
|------|------|---------|
| 2017 | 172  | Partial year (likely starting mid‑year) |
| 2018 | 365  | Full non‑leap year |
| 2019 | 365  | Full non‑leap year |
| 2020 | 366  | Full leap year |
| 2021 | 365  | Full non‑leap year |
| 2022 | 365  | Full non‑leap year |
| 2023 | 365  | Full non‑leap year |
| 2024 | 366  | Full leap year |
| 2025 | 365  | Full non‑leap year (projected or complete) |
| 2026 | 365  | Full non‑leap year (projected or complete) |

**Observations / Наблюдения:**  
- Except for 2017, every year contains either 365 or 366 rows, which matches the number of days in that year.  
- 2017 has only 172 rows – this suggests the data collection started sometime in the middle of the year (around June/July).  
- The total of **3459 rows** equals the sum of all daily records across the 10 years.  
- If you expect every year to be complete, you may want to investigate why 2017 is incomplete.

---

### 2. Columns in the Combined DataFrame  
### 2. Колони в комбинирания DataFrame  

| Column Name | Description (EN) | Описание (BG) |
|-------------|------------------|---------------|
| `Number` | Row index (1..n) within each year | Индекс на реда (1..n) за всяка година |
| `Day` | Date or day identifier | Дата или идентификатор на деня |
| `Steps` | Number of steps taken | Брой стъпки |
| `Difference` | Change in steps from previous day | Промяна в стъпките спрямо предишния ден |
| `Calories` | Calories burned | Изгорени калории |
| `Difference of C` | Change in calories | Промяна в калориите |
| `Distance` | Distance walked (km or miles) | Изминато разстояние (км или мили) |
| `Difference of D` | Change in distance | Промяна в разстоянието |
| `Time` | Active time (minutes) | Активно време (минути) |
| `Difference of time` | Change in active time | Промяна в активното време |

All columns are present in every row (thanks to our standardisation), but many cells contain missing values.

---

### 3. Missing Values Analysis  
### 3. Анализ на липсващите стойности  

| Column | Missing Count | % of total (3459) | Interpretation |
|--------|---------------|-------------------|----------------|
| `Number` | 0 | 0% | Perfect – always present |
| `Day` | 0 | 0% | Perfect – always present |
| `Steps` | 244 | 7.1% | Some days have no step data |
| `Difference` | 11 | 0.3% | Almost complete (only first day of each year missing by design) |
| `Calories` | 416 | 12.0% | Significant missingness |
| `Difference of C` | 182 | 5.3% | Moderate missingness |
| `Distance` | 416 | 12.0% | Same missing pattern as Calories |
| `Difference of D` | 182 | 5.3% | Same pattern as Difference of C |
| `Time` | 416 | 12.0% | Same as Calories and Distance |
| `Difference of time` | 181 | 5.2% | Slightly less than Difference of C |

**Key insights / Основни изводи:**  
- The columns `Calories`, `Distance`, and `Time` have exactly the same number of missing values (416). This likely means that on those days, the entire set of measurements (calories, distance, time) was not recorded.  
- The `Difference` columns (change from previous day) have fewer missing values (between 11 and 182). The 11 missing in `Difference` are probably the first row of each year (since there is no previous day). The other differences (C, D, time) have more missing because they depend on the preceding day's value – if that value is missing, the difference cannot be computed.  
- `Steps` has 244 missing values, which is less than the 416 missing for the other metrics – indicating that sometimes steps were recorded but not calories/distance/time.

**Potential reasons for missing data / Възможни причини за липсващи данни:**  
- Device not worn on some days.  
- Sensor failures or sync issues.  
- Manual entry errors.  

---

### 4. What to do next?  
### 4. Какво да направим по-нататък?  

Depending on your analysis goals, you may consider:

- **Imputation** – fill missing values using interpolation, forward/backward fill, or more advanced methods.  
- **Removal** – drop rows with missing values if they are few (but here ~12% is non‑negligible).  
- **Investigate patterns** – check if missingness is seasonal or correlates with certain days of the week.  
- **Visualise** – plot time series of steps, calories, etc., to see trends and gaps.

**Example code for handling missing values (optional):**  
**Примерен код за обработка на липсващи стойности (по избор):**

```julia
# Drop rows where Calories is missing
df_clean = dropmissing(df_combined, [:Calories, :Distance, :Time])

# Or fill with mean / median
using Statistics
mean_steps = mean(skipmissing(df_combined.Steps))
df_combined.Steps = coalesce.(df_combined.Steps, mean_steps)

# Exporting Combined Data to CSV with Julia  
## Експортиране на комбинираните данни в CSV с Julia  

After combining all sheets into a single DataFrame, we save it as a CSV file using the `CSV.jl` package. This format is natively supported by Julia, can be read back with `CSV.read`, and is compatible with many other Julia packages for visualisation, statistics, and machine learning.

---

### 1. The Export Code  
### 1. Кодът за експорт  

The following code saves the combined DataFrame to a CSV file in the same folder as the original Excel file.

```julia
using CSV  # Ensure the package is loaded (add with Pkg.add("CSV") if needed)

# Define output path (same directory as the Excel file)
output_path = raw"D:\data\projects\steps\Steps_combined.csv"

# Export to CSV
CSV.write(output_path, df_combined)

println("File saved to: $output_path")
println("Total rows exported: ", nrow(df_combined))

In [ ]:
using CSV

# Define output path (same directory as the Excel file)
output_path = raw"D:\data\projects\steps\Steps_combined.csv"

# Export to CSV
CSV.write(output_path, df_combined)

println("File saved to: $output_path")
println("Total rows exported: ", nrow(df_combined))

# Yearly Aggregated Statistics from Steps Data  
## Годишни обобщени статистики от данните за стъпки  

This code performs a comprehensive analysis of your combined steps dataset. It loads the CSV file, parses dates, cleans numeric columns, computes yearly statistics (excluding zero values), prints formatted tables, saves multiple CSV and text outputs, and generates a series of plots—all saved to a local `./output/` folder.

Този код извършва задълбочен анализ на вашия комбиниран набор от данни за стъпки. Той зарежда CSV файла, разпознава датите, почиства числовите колони, изчислява годишни статистики (без нулевите стойности), отпечатва форматирани таблици, записва множество CSV и текстови файлове, и генерира поредица от графики – всичко запазено в локалната папка `./output/`.

---

## 1. Overview of the Code Structure  
## 1. Структура на кода  

The script is organised into eight main parts:

1. **Configure Paths** – set input file and create output folder.  
2. **Load Data** – read the combined CSV into a DataFrame.  
3. **Parse Dates** – extract year and month from various date formats.  
4. **Clean Numeric Columns** – convert columns to `Float64`, handling strings and missing values.  
5. **Compute Yearly Aggregated Statistics** – for each year and each indicator, calculate count, mean, median, std, min, max, and total (excluding zeros).  
6. **Print Formatted Tables** – display clean tables for each metric.  
7. **Export All Outputs** – save full summary, individual metric CSVs, and a text report.  
8. **Generate and Save Plots** – create several visualisations (bar charts with trendlines, comparisons, subplots).

---

## 2. Detailed Explanation of Each Part  
## 2. Подробно обяснение на всяка част  

### Part 1: Configure Paths (Конфигуриране на пътища)  
- The input file path is set to your combined CSV.  
- An output directory `./output/` is created (relative to the script’s current working directory) if it does not already exist. All generated files will be saved there.

### Part 2: Load Data (Зареждане на данните)  
- Uses `CSV.read` to load the CSV into a DataFrame `df`.  
- Prints the total number of rows and column names for verification.

### Part 3: Parse Dates (Разпознаване на датите)  
- Iterates over the `Day` column (which may contain dates in dot format, e.g., `13.7.2017`, or ISO format, e.g., `2017-12-01`).  
- Extracts year and month, handling missing or malformed entries.  
- Adds new columns `Year` and `Month` to the DataFrame.  
- Displays the range of years found.

### Part 4: Clean Numeric Columns (Почистване на числовите колони)  
- Defines `indicator_cols = ["Steps", "Calories", "Distance", "Time"]`.  
- Applies a cleaning function to each column:  
  - If the value is a `Number`, it is converted to `Float64`.  
  - If it is a `String`, it tries to parse it; if the string is `"-"` (placeholder for missing), it returns `missing`.  
  - Any parsing error returns `missing`.  
- Prints the count of non‑missing values for each indicator after cleaning.

### Part 5: Compute Yearly Aggregated Statistics (Изчисляване на годишни обобщени статистики)  
- A helper function `compute_stats` filters out values ≤ 0 and computes: count, mean, median, standard deviation, min, max, and total sum.  
- For each year present in the data, the DataFrame is filtered and stats are computed for each indicator.  
- All results are stored in a `summary_table` DataFrame with clearly named columns (e.g., `Steps_Count`, `Steps_Mean`, …).  
- **Important:** Zero values are excluded from all statistics because they often represent missing or non‑active days.

### Part 6: Print Formatted Tables (Отпечатване на форматирани таблици)  
- Four separate tables are printed in the console (for Steps, Calories, Distance, Time) with renamed columns (e.g., `Valid`, `Mean`, `Median`, `Std`, `Min`, `Max`, `Total`).  
- These tables are easy to read and allow quick inspection of yearly trends.

### Part 7: Export All Outputs (Експортиране на всички резултати)  
- **Full summary CSV** (`yearly_aggregated_stats.csv`) – the entire `summary_table`.  
- **Text report** (`yearly_aggregated_stats.txt`) – contains the same formatted tables plus metadata (timestamp, input file path).  
- **Individual metric CSVs** – for each indicator, a separate CSV with columns: Year, Days_Recorded, Count, Mean, Median, StdDev, Min, Max, Total. These are saved as `steps_yearly.csv`, `calories_yearly.csv`, `distance_yearly.csv`, `time_yearly.csv`.  
- All files are placed in the `./output/` folder.

### Part 8: Generate and Save Plots (Генериране и записване на графики)  
- Uses `Plots.jl` to create eight different plots (PNG format).  
- **Bar charts** for each metric with an overlaid linear trendline (dashed red line).  
- **Multi‑metric comparison** – a single plot with normalised lines to compare all four indicators on the same scale.  
- **2×2 subplot** – a grid of bar charts for all metrics.  
- **Trend line** – a focused plot on the steps trend with regression line.  
- **Stacked bar chart** – normalised metrics shown as adjacent bars.  
- Each plot is saved with a descriptive filename (e.g., `steps_by_year.png`).  
- The plots are also displayed (rendered) in the notebook if you are using Jupyter Lab.

---

## 3. Key Design Choices and Assumptions  
## 3. Основни проектни решения и допускания  

- **Zero values are excluded** because they likely indicate missing data or days with no activity. This ensures the statistics reflect genuine active days.  
- **Date parsing** supports both dot and ISO formats, making the code robust to different Excel exports.  
- **Missing values (`missing`)** are handled gracefully throughout the pipeline; they are excluded from statistical computations.  
- **Linear trendlines** are calculated using a simple linear regression to show whether each metric is increasing or decreasing over the years.  
- **Output folder** is created automatically, avoiding manual directory creation.

---

## 4. Example Output Files  
## 4. Примерни изходни файлове  

After running the script, the `./output/` folder will contain:


In [ ]:
# ============================================================
# STEPS DATA ANALYSIS - YEARLY AGGREGATED STATISTICS
# All indicator columns, no difference columns
# All outputs saved to ./output/ folder
# ============================================================

using CSV
using DataFrames
using Statistics
using Dates

# ============================================================
# CONFIGURE PATHS
# ============================================================

# Input file (adjust this path to your actual CSV location)
input_path = raw"D:\data\projects\steps\Steps_combined.csv"

# Output folder (creates ./output/ relative to where script runs)
output_dir = joinpath(pwd(), "output")
if !isdir(output_dir)
    mkdir(output_dir)
    println("Created output directory: ", output_dir)
else
    println("Using output directory: ", output_dir)
end

# ============================================================
# PART 1: LOAD DATA
# ============================================================

df = CSV.read(input_path, DataFrame)

println("=== Dataset Loaded ===")
println("Total rows: ", nrow(df))
println("Columns: ", names(df))

# ============================================================
# PART 2: PARSE DATES
# ============================================================

years = Int[]
months = Int[]

for d in df.Day
    if !ismissing(d)
        try
            s = string(d)
            # Try dot format: 13.7.2017
            if occursin(".", s) && !occursin("-", s)
                parts = split(s, ".")
                if length(parts) == 3
                    day_num = parse(Int, parts[1])
                    month_num = parse(Int, parts[2])
                    year_num = parse(Int, parts[3])
                    if year_num < 100
                        year_num += 2000
                    end
                    push!(years, year_num)
                    push!(months, month_num)
                else
                    push!(years, 0)
                    push!(months, 0)
                end
            # Try ISO format: 2017-12-01
            else
                dt = Date(s)
                push!(years, year(dt))
                push!(months, month(dt))
            end
        catch
            push!(years, 0)
            push!(months, 0)
        end
    else
        push!(years, 0)
        push!(months, 0)
    end
end

df.Year = years
df.Month = months

println("\n=== Date Parsing ===")
valid_years = sort(unique(filter(y -> y > 0, years)))
println("Years found: ", valid_years)
println("Date range: ", minimum(valid_years), " to ", maximum(valid_years))

# ============================================================
# PART 3: CLEAN NUMERIC COLUMNS
# ============================================================

indicator_cols = ["Steps", "Calories", "Distance", "Time"]

for col in indicator_cols
    if col in names(df)
        df[!, col] = passmissing(x -> begin
            try
                if x isa Number
                    return Float64(x)
                elseif x isa String
                    if strip(x) == "-"
                        return missing
                    end
                    return parse(Float64, x)
                else
                    return missing
                end
            catch
                return missing
            end
        end).(df[!, col])
    end
end

println("\n=== Numeric Columns Cleaned ===")
for col in indicator_cols
    if col in names(df)
        nonmissing = sum(.!ismissing.(df[!, col]))
        println("  ", col, ": ", nonmissing, " non-missing values")
    end
end

# ============================================================
# PART 4: YEARLY AGGREGATED STATISTICS (0 EXCLUDED)
# ============================================================

println("\n" * "="^80)
println("YEARLY AGGREGATED STATISTICS - ALL INDICATORS (0 values excluded)")
println("="^80)

function compute_stats(values::Vector{Float64})
    valid = filter(x -> x > 0, values)
    n = length(valid)
    if n == 0
        return (count=0, mean=0.0, median=0.0, std=0.0, min=0.0, max=0.0, total=0.0)
    end
    return (
        count = n,
        mean = round(mean(valid), digits=1),
        median = round(median(valid), digits=1),
        std = round(std(valid), digits=1),
        min = round(minimum(valid), digits=1),
        max = round(maximum(valid), digits=1),
        total = round(sum(valid), digits=0)
    )
end

# Linear regression helper for trendlines
function linear_fit(x::Vector{Float64}, y::Vector{Float64})
    n = length(x)
    if n < 2
        return (slope=0.0, intercept=0.0)
    end
    x_mean = mean(x)
    y_mean = mean(y)
    numerator = sum((x[i] - x_mean) * (y[i] - y_mean) for i in 1:n)
    denominator = sum((x[i] - x_mean)^2 for i in 1:n)
    if denominator == 0
        return (slope=0.0, intercept=y_mean)
    end
    slope = numerator / denominator
    intercept = y_mean - slope * x_mean
    return (slope=slope, intercept=intercept)
end

years_list = sort(unique(filter(y -> y > 0, df.Year)))

summary_table = DataFrame(
    Year = Int[],
    Days_Recorded = Int[],
    
    Steps_Count = Int[],
    Steps_Mean = Float64[],
    Steps_Median = Float64[],
    Steps_Std = Float64[],
    Steps_Min = Float64[],
    Steps_Max = Float64[],
    Steps_Total = Float64[],
    
    Calories_Count = Int[],
    Calories_Mean = Float64[],
    Calories_Median = Float64[],
    Calories_Std = Float64[],
    Calories_Min = Float64[],
    Calories_Max = Float64[],
    Calories_Total = Float64[],
    
    Distance_Count = Int[],
    Distance_Mean = Float64[],
    Distance_Median = Float64[],
    Distance_Std = Float64[],
    Distance_Min = Float64[],
    Distance_Max = Float64[],
    Distance_Total = Float64[],
    
    Time_Count = Int[],
    Time_Mean = Float64[],
    Time_Median = Float64[],
    Time_Std = Float64[],
    Time_Min = Float64[],
    Time_Max = Float64[],
    Time_Total = Float64[]
)

for yr in years_list
    yr_df = filter(row -> row.Year == yr, df)
    n_days = nrow(yr_df)
    
    steps_vals = Float64.(collect(skipmissing(yr_df.Steps)))
    s = compute_stats(steps_vals)
    
    cal_vals = Float64.(collect(skipmissing(yr_df.Calories)))
    c = compute_stats(cal_vals)
    
    dist_vals = Float64.(collect(skipmissing(yr_df.Distance)))
    d = compute_stats(dist_vals)
    
    time_vals = Float64.(collect(skipmissing(yr_df.Time)))
    t = compute_stats(time_vals)
    
    push!(summary_table, (
        yr, n_days,
        s.count, s.mean, s.median, s.std, s.min, s.max, s.total,
        c.count, c.mean, c.median, c.std, c.min, c.max, c.total,
        d.count, d.mean, d.median, d.std, d.min, d.max, d.total,
        t.count, t.mean, t.median, t.std, t.min, t.max, t.total
    ))
end

# Summary table computed but not printed (see formatted tables below)

# ============================================================
# PART 5: PRINT FORMATTED TABLES
# ============================================================

println("\n" * "="^100)
println("1. STEPS")
println("="^100)
steps_fmt = select(summary_table, 
    :Year, :Days_Recorded => :Days, 
    :Steps_Count => :Valid, :Steps_Mean => :Mean, 
    :Steps_Median => :Median, :Steps_Std => :Std,
    :Steps_Min => :Min, :Steps_Max => :Max, :Steps_Total => :Total)
println(steps_fmt)

println("\n" * "="^100)
println("2. CALORIES")
println("="^100)
cal_fmt = select(summary_table, 
    :Year, :Days_Recorded => :Days,
    :Calories_Count => :Valid, :Calories_Mean => :Mean,
    :Calories_Median => :Median, :Calories_Std => :Std,
    :Calories_Min => :Min, :Calories_Max => :Max, :Calories_Total => :Total)
println(cal_fmt)

println("\n" * "="^100)
println("3. DISTANCE (km)")
println("="^100)
dist_fmt = select(summary_table,
    :Year, :Days_Recorded => :Days,
    :Distance_Count => :Valid, :Distance_Mean => :Mean,
    :Distance_Median => :Median, :Distance_Std => :Std,
    :Distance_Min => :Min, :Distance_Max => :Max, :Distance_Total => :Total)
println(dist_fmt)

println("\n" * "="^100)
println("4. TIME (minutes)")
println("="^100)
time_fmt = select(summary_table,
    :Year, :Days_Recorded => :Days,
    :Time_Count => :Valid, :Time_Mean => :Mean,
    :Time_Median => :Median, :Time_Std => :Std,
    :Time_Min => :Min, :Time_Max => :Max, :Time_Total => :Total)
println(time_fmt)

# ============================================================
# PART 6: EXPORT ALL OUTPUTS TO ./output/ FOLDER
# ============================================================

# 6a: Export full summary CSV
output_csv = joinpath(output_dir, "yearly_aggregated_stats.csv")
CSV.write(output_csv, summary_table)
println("\n=== Exported CSV: ", output_csv, " ===")

# 6b: Export formatted tables as text file
output_txt = joinpath(output_dir, "yearly_aggregated_stats.txt")
open(output_txt, "w") do f
    println(f, "="^100)
    println(f, "YEARLY AGGREGATED STATISTICS - ALL INDICATORS")
    println(f, "Data source: ", input_path)
    println(f, "Generated: ", now())
    println(f, "Note: 0 values excluded from all statistics")
    println(f, "="^100)
    
    println(f, "\n" * "="^100)
    println(f, "1. STEPS")
    println(f, "="^100)
    println(f, steps_fmt)
    
    println(f, "\n" * "="^100)
    println(f, "2. CALORIES")
    println(f, "="^100)
    println(f, cal_fmt)
    
    println(f, "\n" * "="^100)
    println(f, "3. DISTANCE (km)")
    println(f, "="^100)
    println(f, dist_fmt)
    
    println(f, "\n" * "="^100)
    println(f, "4. TIME (minutes)")
    println(f, "="^100)
    println(f, time_fmt)
    
    println(f, "\n" * "="^100)
    println(f, "RAW DATA TABLE (for verification)")
    println(f, "="^100)
    println(f, summary_table)
end
println("=== Exported TXT: ", output_txt, " ===")

# 6c: Export individual metric CSVs
for (col, fname) in [("Steps", "steps_yearly.csv"), ("Calories", "calories_yearly.csv"),
                     ("Distance", "distance_yearly.csv"), ("Time", "time_yearly.csv")]
    col_lower = lowercase(col)
    cols_to_select = [
        :Year, :Days_Recorded,
        Symbol("$(col)_Count"), Symbol("$(col)_Mean"), Symbol("$(col)_Median"),
        Symbol("$(col)_Std"), Symbol("$(col)_Min"), Symbol("$(col)_Max"), Symbol("$(col)_Total")
    ]
    metric_df = select(summary_table, cols_to_select...)
    rename!(metric_df, [
        :Year => :Year, :Days_Recorded => :Days_Recorded,
        Symbol("$(col)_Count") => :Count, Symbol("$(col)_Mean") => :Mean,
        Symbol("$(col)_Median") => :Median, Symbol("$(col)_Std") => :StdDev,
        Symbol("$(col)_Min") => :Min, Symbol("$(col)_Max") => :Max,
        Symbol("$(col)_Total") => :Total
    ])
    out_path = joinpath(output_dir, fname)
    CSV.write(out_path, metric_df)
    println("=== Exported: ", out_path, " ===")
end

# ============================================================
# PART 7: PLOTS (saved to ./output/)
# ============================================================

using Plots

# Plot 1: Average Steps by Year with trendline
x_steps = Float64.(summary_table.Year)
y_steps = summary_table.Steps_Mean
fit_steps = linear_fit(x_steps, y_steps)
trend_steps = fit_steps.slope .* x_steps .+ fit_steps.intercept

p1 = bar(summary_table.Year, summary_table.Steps_Mean,
    title="Average Steps by Year",
    xlabel="Year", ylabel="Average Steps",
    label="Data", color=:steelblue, bar_width=0.6,
    xticks=(summary_table.Year, string.(summary_table.Year)))
plot!(p1, summary_table.Year, trend_steps, 
    label="Trend", color=:red, linewidth=2, linestyle=:dash)
for i in 1:nrow(summary_table)
    annotate!(summary_table.Year[i], summary_table.Steps_Mean[i] + 300, 
        text(string(round(Int, summary_table.Steps_Mean[i])), 8, :center))
end
display(p1)
savefig(p1, joinpath(output_dir, "steps_by_year.png"))
println("=== Saved plot: steps_by_year.png ===")

# Plot 2: Average Calories by Year with trendline
x_cal = Float64.(summary_table.Year)
y_cal = summary_table.Calories_Mean
fit_cal = linear_fit(x_cal, y_cal)
trend_cal = fit_cal.slope .* x_cal .+ fit_cal.intercept

p2 = bar(summary_table.Year, summary_table.Calories_Mean,
    title="Average Calories by Year",
    xlabel="Year", ylabel="Average Calories",
    label="Data", color=:orangered, bar_width=0.6,
    xticks=(summary_table.Year, string.(summary_table.Year)))
plot!(p2, summary_table.Year, trend_cal, 
    label="Trend", color=:red, linewidth=2, linestyle=:dash)
for i in 1:nrow(summary_table)
    annotate!(summary_table.Year[i], summary_table.Calories_Mean[i] + 15,
        text(string(round(Int, summary_table.Calories_Mean[i])), 8, :center))
end
display(p2)
savefig(p2, joinpath(output_dir, "calories_by_year.png"))
println("=== Saved plot: calories_by_year.png ===")

# Plot 3: Average Distance by Year with trendline
x_dist = Float64.(summary_table.Year)
y_dist = summary_table.Distance_Mean
fit_dist = linear_fit(x_dist, y_dist)
trend_dist = fit_dist.slope .* x_dist .+ fit_dist.intercept

p3 = bar(summary_table.Year, summary_table.Distance_Mean,
    title="Average Distance (km) by Year",
    xlabel="Year", ylabel="Average Distance (km)",
    label="Data", color=:forestgreen, bar_width=0.6,
    xticks=(summary_table.Year, string.(summary_table.Year)))
plot!(p3, summary_table.Year, trend_dist, 
    label="Trend", color=:red, linewidth=2, linestyle=:dash)
for i in 1:nrow(summary_table)
    annotate!(summary_table.Year[i], summary_table.Distance_Mean[i] + 0.3,
        text(string(round(summary_table.Distance_Mean[i], digits=1)), 8, :center))
end
display(p3)
savefig(p3, joinpath(output_dir, "distance_by_year.png"))
println("=== Saved plot: distance_by_year.png ===")

# Plot 4: Average Time by Year with trendline
x_time = Float64.(summary_table.Year)
y_time = summary_table.Time_Mean
fit_time = linear_fit(x_time, y_time)
trend_time = fit_time.slope .* x_time .+ fit_time.intercept

p4 = bar(summary_table.Year, summary_table.Time_Mean,
    title="Average Time (min) by Year",
    xlabel="Year", ylabel="Average Time (minutes)",
    label="Data", color=:purple, bar_width=0.6,
    xticks=(summary_table.Year, string.(summary_table.Year)))
plot!(p4, summary_table.Year, trend_time, 
    label="Trend", color=:red, linewidth=2, linestyle=:dash)
for i in 1:nrow(summary_table)
    annotate!(summary_table.Year[i], summary_table.Time_Mean[i] + 3,
        text(string(round(Int, summary_table.Time_Mean[i])), 8, :center))
end
display(p4)
savefig(p4, joinpath(output_dir, "time_by_year.png"))
println("=== Saved plot: time_by_year.png ===")

# Plot 5: Multi-metric comparison (normalized)
p5 = plot(summary_table.Year, summary_table.Steps_Mean ./ 100,
    label="Steps (÷100)", marker=:o, linewidth=2)
plot!(p5, summary_table.Year, summary_table.Calories_Mean ./ 10,
    label="Calories (÷10)", marker=:s, linewidth=2)
plot!(p5, summary_table.Year, summary_table.Distance_Mean .* 100,
    label="Distance (×100)", marker=:d, linewidth=2)
plot!(p5, summary_table.Year, summary_table.Time_Mean,
    label="Time (min)", marker=:x, linewidth=2)
plot!(p5, title="Multi-Metric Comparison by Year",
    xlabel="Year", ylabel="Normalized Values", legend=:outertopright,
    xticks=(summary_table.Year, string.(summary_table.Year)))
display(p5)
savefig(p5, joinpath(output_dir, "multi_metric_comparison.png"))
println("=== Saved plot: multi_metric_comparison.png ===")

# Plot 6: 2x2 subplot of all metrics
p6 = plot(
    bar(summary_table.Year, summary_table.Steps_Mean, 
        title="Steps", color=:steelblue, legend=false,
        xticks=(summary_table.Year, string.(summary_table.Year))),
    bar(summary_table.Year, summary_table.Calories_Mean,
        title="Calories", color=:orangered, legend=false,
        xticks=(summary_table.Year, string.(summary_table.Year))),
    bar(summary_table.Year, summary_table.Distance_Mean,
        title="Distance (km)", color=:forestgreen, legend=false,
        xticks=(summary_table.Year, string.(summary_table.Year))),
    bar(summary_table.Year, summary_table.Time_Mean,
        title="Time (min)", color=:purple, legend=false,
        xticks=(summary_table.Year, string.(summary_table.Year))),
    layout=(2,2), size=(900,700),
    plot_title="All Metrics by Year"
)
display(p6)
savefig(p6, joinpath(output_dir, "all_metrics_by_year.png"))
println("=== Saved plot: all_metrics_by_year.png ===")

# Plot 7: Steps trend line with regression line
x_trend = Float64.(summary_table.Year)
y_trend = summary_table.Steps_Mean
fit_trend = linear_fit(x_trend, y_trend)
trend_line = fit_trend.slope .* x_trend .+ fit_trend.intercept

p7 = plot(
    summary_table.Year, summary_table.Steps_Mean,
    title="Steps Trend by Year", marker=:circle, linewidth=2,
    label="Data", color=:steelblue, xlabel="Year", ylabel="Avg Steps",
    xticks=(summary_table.Year, string.(summary_table.Year))
)
plot!(p7, summary_table.Year, trend_line, 
    label="Trend", color=:red, linewidth=2, linestyle=:dash)
display(p7)
savefig(p7, joinpath(output_dir, "steps_trend.png"))
println("=== Saved plot: steps_trend.png ===")

# Plot 8: Normalized metrics by year — FIXED (no StatsPlots needed)
p8 = plot(title="Normalized Metrics by Year", xlabel="Year", ylabel="Normalized Units",
    legend=:outertopright, xticks=summary_table.Year)

bar_width = 0.2
offsets = [-1.5*bar_width, -0.5*bar_width, 0.5*bar_width, 1.5*bar_width]

bar!(p8, summary_table.Year .+ offsets[1], summary_table.Steps_Mean, 
    label="Steps", color=:steelblue, bar_width=bar_width)
bar!(p8, summary_table.Year .+ offsets[2], summary_table.Calories_Mean .* 10, 
    label="Calories×10", color=:orangered, bar_width=bar_width)
bar!(p8, summary_table.Year .+ offsets[3], summary_table.Distance_Mean .* 1000, 
    label="Distance×1000", color=:forestgreen, bar_width=bar_width)
bar!(p8, summary_table.Year .+ offsets[4], summary_table.Time_Mean .* 50, 
    label="Time×50", color=:purple, bar_width=bar_width)

display(p8)
savefig(p8, joinpath(output_dir, "stacked_metrics_by_year.png"))
println("=== Saved plot: stacked_metrics_by_year.png ===")

# ============================================================
# PART 8: FINAL SUMMARY
# ============================================================

println("\n" * "="^80)
println("COMPLETE! All outputs saved to: ", output_dir)
println("="^80)
println("\nFiles generated:")
println("  CSV files:")
println("    - yearly_aggregated_stats.csv       (full summary)")
println("    - steps_yearly.csv                  (steps only)")
println("    - calories_yearly.csv               (calories only)")
println("    - distance_yearly.csv               (distance only)")
println("    - time_yearly.csv                   (time only)")
println("\n  Text files:")
println("    - yearly_aggregated_stats.txt       (formatted tables)")
println("\n  Plot files:")
println("    - steps_by_year.png")
println("    - calories_by_year.png")
println("    - distance_by_year.png")
println("    - time_by_year.png")
println("    - multi_metric_comparison.png")
println("    - all_metrics_by_year.png")
println("    - steps_trend.png")
println("    - stacked_metrics_by_year.png")
println("="^80)

# Executive Activity & Fitness Report / Доклад за физическата активност
### Multi-Year Longitudinal Analysis (2017–2026) / Многогодишен надлъжен анализ (2017–2026)

---

## 1. Executive Summary & Objective / Резюме и цели

**English:**
This report evaluates your historical physical activity data across a 10-year span to uncover structural shifts in your daily lifestyle habits. By filtering out inactive or unrecorded days ($x = 0$), the analysis focuses entirely on true behavioral trends. The metrics reveal an initial active baseline, a prolonged mid-term decline coinciding with external macro-environmental shocks (e.g., 2020), and an extraordinary physical renaissance culminating in an elite tier of conditioning.

**Български:**
Този доклад оценява вашите исторически данни за физическа активност за период от 10 години, за да открие структурни промени в ежедневните ви навици. Чрез филтриране на дните без активност ($x = 0$), анализът се фокусира изцяло върху реалните поведенчески тенденции. Показателите разкриват първоначална активна базова линия, продължителен спад в средносрочен план, съвпадащ с външни шокове (напр. 2020 г.), и необикновен физически ренесанс, кулминиращ в елитно ниво на кондиция.

---

## 2. Macro-Behavioral Phases / Макроповеденчески фази

### Phase I: The Active Baseline (2018–2019) / Фаза I: Активна изходна база (2018–2019)
* **EN:** Before 2020, daily routines were highly stable and healthy. Daily step averages consistently hovered between $9,800$ and $11,300$ steps, paired with a reliable daily energy expenditure of $\mu_{\text{calories}} \approx 420 \text{ kcal}$.
* **BG:** Преди 2020 г. ежедневието беше високо стабилно и здравословно. Средният дневен брой крачки постоянно варираше между $9,800$ и $11,300$ крачки, съчетан с надежден дневен разход на енергия от $\mu_{\text{calories}} \approx 420 \text{ kcal}$.

### Phase II: The Sedentary Shock (2020) / Фаза II: Застоял шок и волатилност (2020)
* **EN:** The onset of the pandemic caused a sudden contraction in baseline mobility. Average daily step volume dropped by $-42.5\%$ relative to 2019 levels. While the median step count dropped precipitously to an all-time low of $\tilde{x} = 4,054.0 \text{ steps}$, the absolute historical maximum step day occurred during this exact year ($48,003 \text{ steps}$). This points to a highly volatile year with isolated, extreme outdoor events.
* **BG:** Началото на пандемията предизвика внезапно свиване на базовата мобилност. Средният дневен обем на крачките спадна с $-42.5\%$ спрямо нивата от 2019 г. Въпреки че медианата падна до рекордно ниско ниво от $\tilde{x} = 4,054.0 \text{ крачки}$, абсолютният исторически максимум за един ден се случи точно през тази година ($48,003 \text{ крачки}$). Това показва силно волатилна година с изолирани, екстремни събития на открито.

### Phase III: The Non-Linear Recovery (2021–2022) / Фаза III: Нелинейно възстановяване (2021–2022)
* **EN:** Mobility patterns fluctuated significantly. A partial rebound in 2021 ($\mu = 8,582.9 \text{ steps}$) was followed by a secondary contraction in 2022 where daily averages compressed back down to $7,170.0 \text{ steps}$ and a mean distance of $\mu_{\text{dist}} = 5.2 \text{ km}$.
* **BG:** Моделите на мобилност флуктуираха значително. Частично възстановяване през 2021 г. ($\mu = 8,582.9 \text{ крачки}$) беше последвано от вторично свиване през 2022 г., когато дневните средни стойности се свиха обратно до $7,170.0 \text{ крачки}$ и средно разстояние от $\mu_{\text{dist}} = 5.2 \text{ km}$.

### Phase IV: The High-Performance Era (2024–2026) / Фаза IV: Ера на високи постижения (2024–2026)
* **EN:** Beginning in late 2023, your data indicates a deliberate lifestyle shift. 2025 established an all-time high-performance ceiling, maintaining an extraordinary average of $\mu = 14,749.8 \text{ steps}$ and $\mu_{\text{dist}} = 10.6 \text{ km}$ across the entire year. Year-to-date tracking for 2026 confirms that these metrics have stabilized into a permanent lifestyle fixture ($13,046.8 \text{ steps/day}$).
* **BG:** Започвайки в края на 2023 г., вашите данни показват съзнателна промяна в начина на живот. 2025 г. установи исторически таван на производителността, поддържайки необикновена средна стойност от $\mu = 14,749.8 \text{ крачки}$ и $\mu_{\text{dist}} = 10.6 \text{ km}$ през цялата година. Текущите данни за 2026 г. потвърждават, че тези показатели са се стабилизирали в постоянен навик ($13,046.8 \text{ крачки/ден}$).

---

## 3. Multi-Year Fitness Ledger / Многогодишен фитнес регистър

> **Note / Забележка:** All stats exclude zero values. / Всички статистики изключват нулевите стойности.

| Year / Година | Active Days / Активни дни ($N$) | Mean Steps / Средно крачки ($\mu$) | Median Steps / Медиана ($\tilde{x}$) | StdDev / Станд. отклон. ($\sigma$) | Mean Dist / Ср. Разстояние ($\text{km}$) | Mean Calories / Ср. Калории ($\text{kcal}$) |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **2017** | 137 | 6,366.8 | 5,556.0 | 5,392.6 | *N/A* | *N/A* |
| **2018** | 365 | 9,856.7 | 9,580.0 | 4,964.2 | 7.1 | 383.7 |
| **2019** | 357 | 11,227.8 | 11,018.0 | 5,846.5 | 8.1 | 455.0 |
| **2020** | 364 | 6,458.5 | 4,054.0 | 7,262.6 | 4.7 | 276.3 |
| **2021** | 360 | 8,582.9 | 7,526.0 | 5,990.9 | 6.2 | 367.9 |
| **2022** | 361 | 7,170.0 | 6,900.0 | 3,804.9 | 5.2 | 326.2 |
| **2023** | 348 | 9,593.6 | 8,894.5 | 5,187.9 | 6.8 | 427.9 |
| **2024** | 366 | 12,843.5 | 12,403.0 | 6,318.1 | 9.0 | 550.6 |
| **2025** | 365 | **14,749.8** | **14,390.0** | 5,915.2 | **10.6** | **645.0** |
| **2026** | 151 | 13,046.8 | 12,071.0 | 5,328.2 | 9.4 | 580.2 |

---

## 4. Deep-Dive Metric Interpretations / Подробен анализ на показателите

### A. Gait and Distance Expansion / Механика на походката и разстоянието
* **EN:** The linear relationship between average steps and physical distance tracks with absolute precision over the years. Your average stride length has remained exceptionally uniform. In the valley of 2020, you achieved only $4.7\text{ km/day}$, whereas in 2025 you expanded your daily coverage to $10.6\text{ km/day}$—a massive $+125.5\%$ net increase in physical displacement.
* **BG:** Линейната връзка между средния брой крачки и физическото разстояние се проследява с абсолютна точност през годините. Средната ви дължина на крачката е останала изключително еднаква. В дъното през 2020 г. сте постигали само $4.7\text{ km/ден}$, докато през 2025 г. сте разширили ежедневното си покритие до $10.6\text{ km/ден}$ — огромно нетно увеличение от $+125.5\%$ във физическото преместване.

### B. Energy Expenditure Dynamics / Динамика на енергийния разход
* **EN:** Active metabolic output ($\mu_{\text{calories}}$) is strongly driven by distance traveled. The transition from your lowest active year (2020) to your peak year (2025) marks an increase in average intentional caloric burn from $276.3\text{ kcal}$ to $645.0\text{ kcal}$ per day. This substantial $+133.4\%$ change represents an outstanding upgrade in your daily metabolic rate.
* **BG:** Активният метаболитен изход ($\mu_{\text{calories}}$) е силно обвързан с изминатото разстояние. Преходът от най-ниската ви активна година (2020 г.) към пиковата година (2025 г.) бележи увеличение на средното умишлено изгаряне на калории от $276.3\text{ kcal}$ на $645.0\text{ kcal}$ на ден. Тази значителна промяна от $+133.4\%$ представлява изключително подобрение на дневния ви метаболизъм.

### C. Distribution Consistency / Стабилност на разпределението
* **EN:** In 2025, your mean ($\mu = 14,749.8$) vastly outpaced the standard deviation ($\sigma = 5,915.2$). Furthermore, the historical absolute minimum floor shifted from a near-zero $11.0\text{ steps}$ (2021) to a structured minimum of $1,422.0\text{ steps}$ (2025). This proves that your peak performance is a result of **systemic consistency** rather than relying on occasional long walks.
* **BG:** През 2025 г. средната стойност ($\mu = 14,749.8$) значително изпреварва стандартното отклонение ($\sigma = 5,915.2$). Освен това, историческото абсолютно минимално дъно се измести от близко до нулата ниво ($11.0\text{ крачки}$ през 2021 г.) до структуриран минимум от $1,422.0\text{ крачки}$ през 2025 г. Това доказва, че пиковото ви представяне е резултат от **системна последователност**, а не от случайни дълги разходки.

---

## 5. Strategic Conclusions / Стратегически заключения

**English:**
1. **Elite Consistency:** Averaging over 14,000 steps across an entire year places your movement profile in the elite tier of personal health metrics worldwide.
2. **Habit Resilience:** The 2026 data confirms that this is not a short-lived phase. Averaging over 13,000 steps across the first 151 days demonstrates that active mobility has successfully crystallized into a permanent lifestyle design.

**Български:**
1. **Елитно постоянство:** Поддържането на средно ниво над 14 000 крачки през цялата година поставя профила ви на движение в елитния сегмент на показателите за лично здраве в световен мащаб.
2. **Устойчивост на навиците:** Данните за 2026 г. потвърждават, че това не е краткотрайна фаза. Средната стойност от над 13 000 крачки през първите 151 дни показва, че активната мобилност успешно се е превърнала в постоянна част от вашия начин на живот.

# Statistical Modeling for Calories Prediction  
## Статистическо моделиране за прогнозиране на калории  

This script builds and compares multiple regression models to predict daily calories from activity data. It handles severe multicollinearity (Steps, Distance, Time are highly correlated) by using Ridge regression, PCA, and by introducing a derived feature **Intensity = Steps / Time** (steps per minute). The entire analysis is saved to the `./modeling/` folder, with no verbose DataFrame printing to keep the notebook clean.

Този скрипт изгражда и сравнява множество регресионни модели за прогнозиране на дневните калории от данните за активност. Той се справя със силна мултиколинеарност (стъпки, разстояние и време са силно корелирани) чрез използване на Ridge регресия, PCA и чрез въвеждане на производна характеристика **Интензивност = Стъпки / Време** (стъпки в минута). Целият анализ се записва в папка `./modeling/`, без извеждане на големи DataFrames, за да остане бележникът чист.

---

## 1. Configuration and Data Loading  
## 1. Конфигурация и зареждане на данните  

- The input CSV path is set (`Steps_combined.csv`).  
- The output directory `./modeling/` is created automatically.  
- The dataset is loaded and cleaned (date parsing, numeric conversion, removal of missing/zero values).  
- Only complete rows with `Steps > 0`, `Calories > 0`, `Distance > 0`, `Time > 0` are retained.

---

## 2. Derived Feature: Intensity  
## 2. Производна характеристика: Интензивност  

A new variable **`Intensity = Steps / Time`** is added. It represents the average step rate (steps per minute), which captures the **pace** of walking/running. This helps separate the **volume** (total steps) from the **intensity** (rate) – two physiologically distinct contributors to calorie burn.

Добавя се нова променлива **`Intensity = Steps / Time`**. Тя представлява средният темп на стъпки (стъпки в минута), който улавя **интензивността** на ходене/бягане. Това помага да се разделят **обемът** (общ брой стъпки) от **интензивността** (темп) – два физиологично различни фактора за изразходване на калории.

---

## 3. Exploratory Data Analysis (EDA)  
## 3. Изследователски анализ на данните (EDA)  

- **Descriptive statistics** (mean, median, std, min, max, skewness, kurtosis) for all variables.  
- **Correlation matrix** – strong correlations between Steps, Distance, and Time (often >0.9).  
- **Pairwise scatter plots** to visualise relationships.  
- **Heatmap** of correlation matrix saved as `correlation_heatmap.png`.  
- **Pairwise scatter matrix** saved as `pairwise_scatter.png`.

---

## 4. Distribution Analysis & Normality  
## 4. Анализ на разпределението и нормалност  

- QQ plots and histograms with normal density curves for all predictors and response.  
- Jarque‑Bera test for normality.  
- Log‑transformed versions are also assessed – many variables become more normal after log transformation, justifying the log‑log model.  
- Plots saved as `qq_plots.png`, `histograms.png`, and `log_histograms.png`.

---

## 5. Multicollinearity Diagnostics  
## 5. Диагностика на мултиколинеарност  

- **Variance Inflation Factor (VIF)** is computed for the original set (`Steps`, `Distance`, `Time`) and for the set with the derived feature (`Steps`, `Time`, `Intensity`).  
- Original predictors show VIF > 10 (severe multicollinearity).  
- After adding Intensity, VIFs drop significantly, confirming that Intensity helps decorrelate the predictors.  
- Condition numbers are also reported.

---

## 6. Modeling Approaches  
## 6. Подходи за моделиране  

The script fits nine different models:

1. **Linear OLS** – `Calories ~ Steps + Distance + Time` (full original).  
2. **Log‑Log** – `log(Calories) ~ log(Steps) + log(Distance) + log(Time)`.  
3. **Reduced** – `Calories ~ Steps + Time` (removes Distance to reduce collinearity).  
4. **Ridge Regression** – with λ = 1.0 on original predictors.  
5. **PCA Regression** – using all principal components (3).  
6. **PCA‑2** – using only the first two principal components (if available).  
7. **Steps + Intensity** – `Calories ~ Steps + Intensity`.  
8. **Time + Intensity** – `Calories ~ Time + Intensity`.  
9. **Intensity Only** – `Calories ~ Intensity` (simplest model).

All models are saved as text summaries in the `./modeling/` folder (e.g., `model1_summary.txt`, etc.).

---

## 7. Model Diagnostics and Residual Analysis  
## 7. Диагностика на моделите и анализ на остатъците  

- For each model, R², adjusted R², RMSE, MAE, AIC, BIC are computed.  
- **Residual analysis** is performed on the Reduced Model (Model 3):  
  - Residuals vs fitted values.  
  - Normal Q‑Q plot.  
  - Histogram of residuals with normal curve.  
  - Scale‑location plot (√|standardized residuals| vs fitted).  
- Jarque‑Bera test on residuals checks normality.  
- Residual diagnostic plots saved as `residual_diagnostics.png`.

---

## 8. Model Comparison Table  
## 8. Таблица за сравнение на моделите  

A comprehensive comparison table is printed and saved as `model_comparison.csv`, showing R², adjusted R², RMSE, MAE, AIC, and BIC for each model. This allows direct comparison of performance.

---

## 9. Feature Importance  
## 9. Значимост на характеристиките  

Standardized coefficients are computed for the Reduced Model (Steps + Time) and for the Steps + Intensity model. This reveals the relative importance of each predictor. Saved as `feature_importance.csv`.

---

## 10. Prediction Validation (Train/Test Split)  
## 10. Валидация на прогнози (разделяне на обучение/тест)  

- 80% of data is used for training, 20% for testing.  
- Two models are retrained on the training set: Reduced (Steps+Time) and Steps+Intensity.  
- Test set metrics: RMSE, MAE, MAPE are computed.  
- A scatter plot of actual vs predicted calories for the Steps+Intensity model is saved as `prediction_accuracy.png`.

---

## 11. Final Report and Outputs  
## 11. Финален доклад и изходни файлове  

- A **modeling report** (`modeling_report.txt`) summarises the dataset, multicollinearity findings, new models with Intensity, model comparison, and recommendations.  
- All model coefficients are collected into `model_coefficients.csv`.  
- The complete output folder contains:


In [ ]:
# ============================================================
# STATISTICAL MODELING - CALORIES PREDICTION
# Handles severe multicollinearity via Ridge + PCA + Reduced models
# + DERIVED FEATURE: Intensity = Steps / Time
# NO verbose DataFrame printing – everything saved to ./modeling/
# ============================================================

using CSV
using DataFrames
using Statistics
using Dates
using LinearAlgebra
using GLM
using StatsBase
using Distributions
using Plots
using StatsPlots
using HypothesisTests
using MultivariateStats
using Measures
using Colors

# ============================================================
# CONFIGURE PATHS
# ============================================================

input_path = raw"D:\data\projects\steps\Steps_combined.csv"
output_dir = joinpath(pwd(), "modeling")
if !isdir(output_dir)
    mkdir(output_dir)
    println("Created modeling directory: ", output_dir)
else
    println("Using modeling directory: ", output_dir)
end

# ============================================================
# PART 1: LOAD & CLEAN DAILY DATA
# ============================================================

df_raw = CSV.read(input_path, DataFrame)
println("=== Dataset Loaded ===")
println("Total rows: ", nrow(df_raw))
println("Columns: ", names(df_raw))

# Parse dates
years = Int[]
months = Int[]
days_parsed = Date[]

for d in df_raw.Day
    if !ismissing(d)
        try
            s = string(d)
            if occursin(".", s) && !occursin("-", s)
                parts = split(s, ".")
                if length(parts) == 3
                    day_num = parse(Int, parts[1])
                    month_num = parse(Int, parts[2])
                    year_num = parse(Int, parts[3])
                    if year_num < 100
                        year_num += 2000
                    end
                    push!(years, year_num)
                    push!(months, month_num)
                    push!(days_parsed, Date(year_num, month_num, day_num))
                else
                    push!(years, 0); push!(months, 0); push!(days_parsed, Date(1,1,1))
                end
            else
                dt = Date(s)
                push!(years, year(dt))
                push!(months, month(dt))
                push!(days_parsed, dt)
            end
        catch
            push!(years, 0); push!(months, 0); push!(days_parsed, Date(1,1,1))
        end
    else
        push!(years, 0); push!(months, 0); push!(days_parsed, Date(1,1,1))
    end
end

df_raw.Year = years
df_raw.Month = months
df_raw.Date = days_parsed

# Clean numeric columns
indicator_cols = ["Steps", "Calories", "Distance", "Time"]

for col in indicator_cols
    if col in names(df_raw)
        df_raw[!, col] = passmissing(x -> begin
            try
                if x isa Number
                    return Float64(x)
                elseif x isa String
                    if strip(x) == "-"
                        return missing
                    end
                    return parse(Float64, x)
                else
                    return missing
                end
            catch
                return missing
            end
        end).(df_raw[!, col])
    end
end

# ============================================================
# PART 2: CREATE MODELING DATASET & DERIVED INTENSITY FEATURE
# ============================================================

df_model = dropmissing(df_raw[:, [:Date, :Year, :Month, :Steps, :Calories, :Distance, :Time]])
df_model = filter(row -> row.Steps > 0 && row.Calories > 0 && row.Distance > 0 && row.Time > 0, df_model)

# Create intensity metric: steps per minute (pace proxy)
df_model.Intensity = df_model.Steps ./ df_model.Time

println("\n=== Modeling Dataset ===")
println("Valid observations: ", nrow(df_model))
println("Date range: ", minimum(df_model.Date), " to ", maximum(df_model.Date))
println("Years: ", sort(unique(df_model.Year)))

# Compact summary
println("\n--- Data Summary ---")
for col in [:Steps, :Calories, :Distance, :Time, :Intensity]
    vals = df_model[!, col]
    println("$(col): mean=$(round(mean(vals),digits=1)), std=$(round(std(vals),digits=1)), range=[$(round(minimum(vals),digits=1)), $(round(maximum(vals),digits=1))]")
end

# ============================================================
# PART 3: EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

println("\n" * "="^80)
println("EXPLORATORY DATA ANALYSIS")
println("="^80)

# 3a: Descriptive Statistics
println("\n--- Descriptive Statistics ---")
for col in [:Steps, :Calories, :Distance, :Time, :Intensity]
    vals = df_model[!, col]
    println("\n$(col):")
    println("  Mean:   ", round(mean(vals), digits=2))
    println("  Median: ", round(median(vals), digits=2))
    println("  Std:    ", round(std(vals), digits=2))
    println("  Min:    ", round(minimum(vals), digits=2))
    println("  Max:    ", round(maximum(vals), digits=2))
    println("  Skew:   ", round(skewness(vals), digits=3))
    println("  Kurt:   ", round(kurtosis(vals), digits=3))
end

# 3b: Correlation Analysis (including Intensity)
println("\n--- Correlation Matrix ---")
corr_vars = [:Steps, :Calories, :Distance, :Time, :Intensity]
corr_labels = ["Steps", "Calories", "Distance", "Time", "Intensity"]
corr_matrix = cor(Matrix(df_model[:, corr_vars]))

# Compact print
println("         Steps   Calor   Dist    Time    Intens")
for i in 1:5
    print(rpad(corr_labels[i], 8))
    for j in 1:5
        print(rpad(round(corr_matrix[i,j], digits=3), 8))
    end
    println()
end

# Save
corr_df = DataFrame(corr_matrix, corr_labels)
insertcols!(corr_df, 1, :Variable => corr_labels)
CSV.write(joinpath(output_dir, "correlation_matrix.csv"), corr_df)

# Heatmap
p_corr = heatmap(1:5, 1:5, corr_matrix, 
    xticks=(1:5, corr_labels), yticks=(1:5, corr_labels),
    title="Correlation Matrix", 
    color=cgrad([:darkblue, :lightblue, :white, :orange, :darkred]),
    clim=(-1,1),
    size=(800,700),
    colorbar_title="Correlation")

for i in 1:5, j in 1:5
    text_color = abs(corr_matrix[i,j]) > 0.6 ? :white : :black
    annotate!(p_corr, j, i, text(round(corr_matrix[i,j], digits=3), 11, text_color, :center))
end

savefig(p_corr, joinpath(output_dir, "correlation_heatmap.png"))
display(p_corr)
println("Saved: correlation_heatmap.png")

# 3c: Pairwise scatter plots (all variables)
p_pairs = plot(layout=(5,5), size=(1200,1200), plot_title="Pairwise Scatter Plot Matrix")

for (i, col_i) in enumerate(corr_vars)
    for (j, col_j) in enumerate(corr_vars)
        subplot_idx = (i-1)*5 + j
        if i == j
            plot!(p_pairs, subplot=subplot_idx, df_model[!, col_i], 
                seriestype=:histogram, bins=40, color=:steelblue, alpha=0.7,
                title="$(corr_labels[i])", legend=false,
                xtickfontsize=6, ytickfontsize=6)
        elseif i > j
            plot!(p_pairs, subplot=subplot_idx, 
                df_model[!, col_j], df_model[!, col_i],
                seriestype=:scatter, color=:steelblue, markersize=2, alpha=0.4,
                legend=false, xtickfontsize=6, ytickfontsize=6)
        else
            plot!(p_pairs, subplot=subplot_idx, 
                [], [],
                annotations=(0.5, 0.5, text("r=$(round(corr_matrix[i,j], digits=3))", 14, :center)),
                legend=false, xaxis=false, yaxis=false, border=:none)
        end
    end
end

savefig(p_pairs, joinpath(output_dir, "pairwise_scatter.png"))
display(p_pairs)
println("Saved: pairwise_scatter.png")

# ============================================================
# PART 4: DISTRIBUTION ANALYSIS & QQ PLOTS
# ============================================================

println("\n" * "="^80)
println("DISTRIBUTION ANALYSIS & NORMALITY TESTS")
println("="^80)

function qq_plot_data(data::Vector{Float64})
    n = length(data)
    sorted_data = sort(data)
    theoretical_quantiles = quantile.(Normal(0,1), (1:n .- 0.5) ./ n)
    return (theoretical_quantiles, sorted_data)
end

fig_qq = plot(layout=(2,3), size=(1200,900), plot_title="QQ Plots - Normality Assessment")
fig_hist = plot(layout=(2,3), size=(1200,900), plot_title="Histograms with Density Curves")

for (i, col) in enumerate([:Steps, :Calories, :Distance, :Time, :Intensity])
    vals = collect(df_model[!, col])
    label = string(col)

    # QQ Plot
    tq, sq = qq_plot_data(vals)
    plot!(fig_qq, subplot=i, tq, sq, seriestype=:scatter, 
        label="Data", color=:steelblue, markersize=3,
        xlabel="Theoretical Quantiles", ylabel="Sample Quantiles",
        title="$(label)")
    mu, sigma = mean(vals), std(vals)
    plot!(fig_qq, subplot=i, tq, mu .+ sigma .* tq, 
        label="Normal Line", color=:red, linewidth=2, linestyle=:dash)

    # Histogram with density
    plot!(fig_hist, subplot=i, vals, seriestype=:histogram, 
        normalize=:pdf, bins=50, color=:steelblue, alpha=0.6,
        label="Histogram", title="$(label)", xlabel="Value", ylabel="Density")
    x_range = range(minimum(vals), maximum(vals), length=200)
    plot!(fig_hist, subplot=i, x_range, pdf.(Normal(mu, sigma), x_range),
        label="Normal", color=:red, linewidth=2)

    jb_stat = JarqueBeraTest(vals)
    jb_pval = HypothesisTests.pvalue(jb_stat)
    println("\n$(label) Normality (Jarque-Bera):")
    println("  Statistic: ", round(jb_stat.JB, digits=2))
    println("  p-value:   ", round(jb_pval, digits=6))
    println("  Normal?    ", jb_pval > 0.05 ? "YES (fail to reject)" : "NO (reject)")
end

savefig(fig_qq, joinpath(output_dir, "qq_plots.png"))
display(fig_qq)
println("\nSaved: qq_plots.png")
savefig(fig_hist, joinpath(output_dir, "histograms.png"))
display(fig_hist)
println("Saved: histograms.png")

# ============================================================
# PART 5: LOG TRANSFORMATION ASSESSMENT
# ============================================================

println("\n" * "="^80)
println("LOG-TRANSFORMED DISTRIBUTIONS")
println("="^80)

fig_log = plot(layout=(2,3), size=(1200,900), plot_title="Log-Transformed Histograms")

for (i, col) in enumerate([:Steps, :Calories, :Distance, :Time, :Intensity])
    vals = collect(df_model[!, col])
    log_vals = log.(vals)
    label = string(col)

    plot!(fig_log, subplot=i, log_vals, seriestype=:histogram,
        normalize=:pdf, bins=50, color=:forestgreen, alpha=0.6,
        label="Log-Data", title="log($(label))", xlabel="log(Value)", ylabel="Density")

    mu_log, sigma_log = mean(log_vals), std(log_vals)
    x_range = range(minimum(log_vals), maximum(log_vals), length=200)
    plot!(fig_log, subplot=i, x_range, pdf.(Normal(mu_log, sigma_log), x_range),
        label="Normal", color=:red, linewidth=2)

    jb_log = JarqueBeraTest(log_vals)
    jb_log_pval = HypothesisTests.pvalue(jb_log)
    println("\nlog($(label)) Normality (Jarque-Bera):")
    println("  p-value: ", round(jb_log_pval, digits=6))
    println("  Improved? ", jb_log_pval > 0.05 ? "YES - Use log transform" : "Still non-normal")
end

savefig(fig_log, joinpath(output_dir, "log_histograms.png"))
display(fig_log)
println("\nSaved: log_histograms.png")

# ============================================================
# PART 6: MULTICOLLINEARITY DIAGNOSTICS
# ============================================================

println("\n" * "="^80)
println("MULTICOLLINEARITY DIAGNOSTICS")
println("="^80)

function compute_vif(X::Matrix{Float64}, j::Int)
    y_vif = X[:, j]
    X_others = X[:, setdiff(1:size(X,2), j)]
    beta_vif = X_others \ y_vif
    y_pred_vif = X_others * beta_vif
    ss_res = sum((y_vif .- y_pred_vif).^2)
    ss_tot = sum((y_vif .- mean(y_vif)).^2)
    r2_vif = 1 - ss_res / ss_tot
    return 1.0 / (1.0 - r2_vif)
end

# Original set
X_raw = Matrix(df_model[:, [:Steps, :Distance, :Time]])
predictor_names_orig = ["Steps", "Distance", "Time"]
vif_orig = [compute_vif(X_raw, j) for j in 1:3]
println("\n--- Original predictors ---")
for (name, vif) in zip(predictor_names_orig, vif_orig)
    flag = vif > 10 ? " *** SEVERE ***" : (vif > 5 ? " ** MODERATE **" : " OK")
    println("  $(rpad(name, 10)): $(round(vif, digits=2))$(flag)")
end

# Reduced set: Steps, Time, Intensity
X_red = Matrix(df_model[:, [:Steps, :Time, :Intensity]])
pred_red = ["Steps", "Time", "Intensity"]
vif_red = [compute_vif(X_red, j) for j in 1:3]
println("\n--- Reduced + Intensity set ---")
for (name, vif) in zip(pred_red, vif_red)
    flag = vif > 10 ? " *** SEVERE ***" : (vif > 5 ? " ** MODERATE **" : " OK")
    println("  $(rpad(name, 10)): $(round(vif, digits=2))$(flag)")
end

# Condition numbers
for (name, X) in [("Original", X_raw), ("Reduced+Intensity", X_red)]
    Xc = X .- mean(X, dims=1)
    Xs = Xc ./ std(Xc, dims=1)
    sv = svd(Xs).S
    cn = maximum(sv) / minimum(sv)
    println("\nCondition Number ($name): $(round(cn, digits=1))")
end

# ============================================================
# PART 7: MULTIPLE REGRESSION MODELS
# ============================================================

println("\n" * "="^80)
println("MULTIPLE REGRESSION MODELS")
println("="^80)

function save_model_summary(model, filepath, model_name)
    open(filepath, "w") do io
        println(io, "Model: $model_name")
        println(io, "Formula: ", formula(model))
        println(io, "Coefficients:")
        ct = coeftable(model)
        println(io, ct)
        r2_val = r2(model)
        adjr2_val = adjr2(model)
        aic_val = aic(model)
        bic_val = bic(model)
        println(io, "R²: ", round(r2_val, digits=6))
        println(io, "Adj. R²: ", round(adjr2_val, digits=6))
        println(io, "AIC: ", round(aic_val, digits=3))
        println(io, "BIC: ", round(bic_val, digits=3))
    end
    println("  -> saved to $filepath")
end

# Model 1: Linear OLS (full original)
println("\n--- Model 1: Linear OLS (Calories ~ Steps + Distance + Time) ---")
formula1 = @formula(Calories ~ Steps + Distance + Time)
model1 = lm(formula1, df_model)
save_model_summary(model1, joinpath(output_dir, "model1_summary.txt"), "Linear OLS Full")

# Model 2: Log-log
println("\n--- Model 2: Log-Log ---")
df_log = DataFrame(
    log_Calories = log.(df_model.Calories),
    log_Steps = log.(df_model.Steps),
    log_Distance = log.(df_model.Distance),
    log_Time = log.(df_model.Time),
    Year = df_model.Year,
    Month = df_model.Month
)
formula2 = @formula(log_Calories ~ log_Steps + log_Distance + log_Time)
model2 = lm(formula2, df_log)
save_model_summary(model2, joinpath(output_dir, "model2_summary.txt"), "Log-Log")

# Model 3: Reduced (Steps + Time)
println("\n--- Model 3: Reduced Model (Calories ~ Steps + Time) ---")
formula3 = @formula(Calories ~ Steps + Time)
model3 = lm(formula3, df_model)
save_model_summary(model3, joinpath(output_dir, "model3_summary.txt"), "Reduced")

# Model 4: Ridge
println("\n--- Model 4: Ridge Regression (lambda=1.0) ---")
function ridge_regression(X::Matrix{Float64}, y::Vector{Float64}, lambda::Float64)
    n, p = size(X)
    X_mean = mean(X, dims=1)
    X_std = std(X, dims=1)
    X_scaled = (X .- X_mean) ./ X_std
    y_mean = mean(y)
    y_centered = y .- y_mean
    beta_ridge = (X_scaled'X_scaled + lambda * I(p)) \ (X_scaled'y_centered)
    beta_original = beta_ridge ./ vec(X_std)
    intercept = y_mean - sum(beta_original .* vec(X_mean))
    y_pred = X * beta_original .+ intercept
    residuals = y .- y_pred
    return (beta=beta_original, intercept=intercept, y_pred=y_pred, residuals=residuals)
end

X_ridge = Matrix(df_model[:, [:Steps, :Distance, :Time]])
y_ridge = df_model.Calories
ridge_result = ridge_regression(X_ridge, y_ridge, 1.0)

ridge_ss_res = sum(ridge_result.residuals.^2)
ridge_ss_tot = sum((y_ridge .- mean(y_ridge)).^2)
ridge_r2 = 1 - ridge_ss_res / ridge_ss_tot
ridge_rmse = sqrt(mean(ridge_result.residuals.^2))

println("Ridge Coefficients (lambda=1.0):")
println("  Intercept: $(round(ridge_result.intercept, digits=4))")
for (name, coef) in zip(predictor_names_orig, ridge_result.beta)
    println("  $(rpad(name, 10)): $(round(coef, digits=6))")
end
println("  R²:   $(round(ridge_r2, digits=4))")
println("  RMSE: $(round(ridge_rmse, digits=2))")

# Model 5 & 6: PCA
println("\n--- Model 5/6: PCA Regression ---")
X_pca = Matrix(df_model[:, [:Steps, :Distance, :Time]])'
pca_model = fit(PCA, X_pca; maxoutdim=3)
pca_vars = principalvars(pca_model)
ncomp = length(pca_vars)
total_var = tprincipalvar(pca_model)
explained_var = pca_vars ./ total_var
println("PCA Explained Variance (based on $(ncomp) actual component(s)):")
for i in 1:ncomp
    println("  PC$(i): $(round(explained_var[i]*100, digits=1))%")
end
pca_scores = MultivariateStats.transform(pca_model, X_pca)'
pc_cols = [Symbol("PC$(i)") for i in 1:ncomp]
df_pca = DataFrame(Calories = df_model.Calories)
for (i, col_sym) in enumerate(pc_cols)
    df_pca[!, col_sym] = pca_scores[:, i]
end
formula_all_pcs = Term(:Calories) ~ sum(Term.(pc_cols))
model5 = lm(formula_all_pcs, df_pca)
save_model_summary(model5, joinpath(output_dir, "model5_summary.txt"), "PCA-$(ncomp) Components")
model6 = nothing
if ncomp >= 2
    formula6 = @formula(Calories ~ PC1 + PC2)
    model6 = lm(formula6, df_pca)
    save_model_summary(model6, joinpath(output_dir, "model6_summary.txt"), "PCA-2 Components")
else
    println("  -> Only $(ncomp) principal component(s) available; PCA-2 model skipped.")
end

# ---------- NEW MODELS WITH DERIVED INTENSITY FEATURE ----------
println("\n--- NEW MODELS: Derived Intensity (Steps/Time) ---")

# Model 7: Steps + Intensity (volume + pace)
formula7 = @formula(Calories ~ Steps + Intensity)
model7 = lm(formula7, df_model)
save_model_summary(model7, joinpath(output_dir, "model7_summary.txt"), "Steps + Intensity")

# Model 8: Time + Intensity (duration + pace)
formula8 = @formula(Calories ~ Time + Intensity)
model8 = lm(formula8, df_model)
save_model_summary(model8, joinpath(output_dir, "model8_summary.txt"), "Time + Intensity")

# Model 9: Intensity alone (simple physiology)
formula9 = @formula(Calories ~ Intensity)
model9 = lm(formula9, df_model)
save_model_summary(model9, joinpath(output_dir, "model9_summary.txt"), "Intensity Only")

# ============================================================
# PART 8: MODEL DIAGNOSTICS
# ============================================================

println("\n" * "="^80)
println("MODEL DIAGNOSTICS")
println("="^80)

function model_diagnostics(model, df, y_col, model_name)
    y_actual = df[!, y_col]
    y_pred = predict(model)
    residuals = y_actual .- y_pred
    n = length(residuals)
    p = length(coef(model))
    ss_res = sum(residuals.^2)
    ss_tot = sum((y_actual .- mean(y_actual)).^2)
    r2_val = 1 - ss_res / ss_tot
    adj_r2 = 1 - (ss_res / (n - p)) / (ss_tot / (n - 1))
    rmse = sqrt(mean(residuals.^2))
    mae = mean(abs.(residuals))
    aic_val = n * log(ss_res / n) + 2 * p
    bic_val = n * log(ss_res / n) + p * log(n)
    println("\n$(model_name):")
    println("  R²:        ", round(r2_val, digits=4))
    println("  Adj R²:    ", round(adj_r2, digits=4))
    println("  RMSE:      ", round(rmse, digits=2))
    println("  MAE:       ", round(mae, digits=2))
    println("  AIC:       ", round(aic_val, digits=2))
    println("  BIC:       ", round(bic_val, digits=2))
    return (r2=r2_val, adj_r2=adj_r2, rmse=rmse, mae=mae, aic=aic_val, bic=bic_val,
            residuals=residuals, y_pred=y_pred)
end

diag1 = model_diagnostics(model1, df_model, :Calories, "Model 1: Linear OLS (full)")
diag2 = model_diagnostics(model2, df_log, :log_Calories, "Model 2: Log-Log")
diag3 = model_diagnostics(model3, df_model, :Calories, "Model 3: Reduced (Steps+Time)")
diag5 = model_diagnostics(model5, df_pca, :Calories, "Model 5: PCA-$(ncomp) Components")
diag6 = nothing
if model6 !== nothing
    diag6 = model_diagnostics(model6, df_pca, :Calories, "Model 6: PCA-2 Components")
end
diag7 = model_diagnostics(model7, df_model, :Calories, "Model 7: Steps + Intensity")
diag8 = model_diagnostics(model8, df_model, :Calories, "Model 8: Time + Intensity")
diag9 = model_diagnostics(model9, df_model, :Calories, "Model 9: Intensity Only")

# ============================================================
# PART 9: RESIDUAL ANALYSIS (Reduced Model)
# ============================================================

println("\n" * "="^80)
println("RESIDUAL ANALYSIS - Model 3 (Reduced)")
println("="^80)

fig_resid = plot(layout=(2,2), size=(1000,900), plot_title="Residual Diagnostics - Reduced Model")
plot!(fig_resid, subplot=1, diag3.y_pred, diag3.residuals, seriestype=:scatter,
    color=:steelblue, markersize=3, xlabel="Fitted Values", ylabel="Residuals",
    title="Residuals vs Fitted")
plot!(fig_resid, subplot=1, [minimum(diag3.y_pred), maximum(diag3.y_pred)], [0,0],
    color=:red, linewidth=2, linestyle=:dash, label="Zero Line")
tq_r, sq_r = qq_plot_data(diag3.residuals)
plot!(fig_resid, subplot=2, tq_r, sq_r, seriestype=:scatter,
    color=:steelblue, markersize=3, xlabel="Theoretical Quantiles", ylabel="Residual Quantiles",
    title="Normal Q-Q (Residuals)")
mu_r, sigma_r = mean(diag3.residuals), std(diag3.residuals)
plot!(fig_resid, subplot=2, tq_r, mu_r .+ sigma_r .* tq_r,
    color=:red, linewidth=2, linestyle=:dash, label="Normal Line")
plot!(fig_resid, subplot=3, diag3.residuals, seriestype=:histogram,
    normalize=:pdf, bins=50, color=:steelblue, alpha=0.6,
    xlabel="Residuals", ylabel="Density", title="Residual Distribution")
x_range_r = range(minimum(diag3.residuals), maximum(diag3.residuals), length=200)
plot!(fig_resid, subplot=3, x_range_r, pdf.(Normal(mu_r, sigma_r), x_range_r),
    color=:red, linewidth=2, label="Normal")
std_residuals = diag3.residuals ./ std(diag3.residuals)
plot!(fig_resid, subplot=4, diag3.y_pred, sqrt.(abs.(std_residuals)), seriestype=:scatter,
    color=:steelblue, markersize=3, xlabel="Fitted Values", 
    ylabel="√|Standardized Residuals|", title="Scale-Location")
savefig(fig_resid, joinpath(output_dir, "residual_diagnostics.png"))
display(fig_resid)
println("Saved: residual_diagnostics.png")

jb_resid = JarqueBeraTest(diag3.residuals)
jb_resid_pval = HypothesisTests.pvalue(jb_resid)
println("\nResidual Normality (Jarque-Bera):")
println("  Statistic: ", round(jb_resid.JB, digits=2))
println("  p-value:   ", round(jb_resid_pval, digits=6))
println("  Normal?    ", jb_resid_pval > 0.05 ? "YES" : "NO")

# ============================================================
# PART 10: MODEL COMPARISON TABLE
# ============================================================

println("\n" * "="^80)
println("MODEL COMPARISON SUMMARY")
println("="^80)

comp_models = ["Linear OLS (full)", "Log-Log", "Reduced (Steps+Time)", "Ridge (lambda=1)",
               "PCA-$(ncomp)", (model6 !== nothing ? "PCA-2" : nothing),
               "Steps+Intensity", "Time+Intensity", "Intensity Only"]
comp_models = filter(x -> x !== nothing, comp_models)

comp_r2   = [diag1.r2, diag2.r2, diag3.r2, ridge_r2, diag5.r2]
comp_adj  = [diag1.adj_r2, diag2.adj_r2, diag3.adj_r2, ridge_r2, diag5.adj_r2]
comp_rmse = [diag1.rmse, diag2.rmse, diag3.rmse, ridge_rmse, diag5.rmse]
comp_mae  = [diag1.mae, diag2.mae, diag3.mae, mean(abs.(ridge_result.residuals)), diag5.mae]
comp_aic  = [diag1.aic, diag2.aic, diag3.aic, NaN, diag5.aic]
comp_bic  = [diag1.bic, diag2.bic, diag3.bic, NaN, diag5.bic]

if model6 !== nothing
    push!(comp_r2, diag6.r2); push!(comp_adj, diag6.adj_r2); push!(comp_rmse, diag6.rmse)
    push!(comp_mae, diag6.mae); push!(comp_aic, diag6.aic); push!(comp_bic, diag6.bic)
end
push!(comp_r2, diag7.r2); push!(comp_adj, diag7.adj_r2); push!(comp_rmse, diag7.rmse)
push!(comp_mae, diag7.mae); push!(comp_aic, diag7.aic); push!(comp_bic, diag7.bic)
push!(comp_r2, diag8.r2); push!(comp_adj, diag8.adj_r2); push!(comp_rmse, diag8.rmse)
push!(comp_mae, diag8.mae); push!(comp_aic, diag8.aic); push!(comp_bic, diag8.bic)
push!(comp_r2, diag9.r2); push!(comp_adj, diag9.adj_r2); push!(comp_rmse, diag9.rmse)
push!(comp_mae, diag9.mae); push!(comp_aic, diag9.aic); push!(comp_bic, diag9.bic)

comparison = DataFrame(
    Model = comp_models,
    R2 = comp_r2,
    Adj_R2 = comp_adj,
    RMSE = comp_rmse,
    MAE = comp_mae,
    AIC = comp_aic,
    BIC = comp_bic
)
println(comparison)
CSV.write(joinpath(output_dir, "model_comparison.csv"), comparison)
println("Saved: model_comparison.csv")

# ============================================================
# PART 11: FEATURE IMPORTANCE (Reduced Model)
# ============================================================

println("\n" * "="^80)
println("FEATURE IMPORTANCE")
println("="^80)

function standardize(x)
    return (x .- mean(x)) ./ std(x)
end

df_std = DataFrame(
    Calories_std = standardize(df_model.Calories),
    Steps_std = standardize(df_model.Steps),
    Time_std = standardize(df_model.Time),
    Intensity_std = standardize(df_model.Intensity)
)
model_std3 = lm(@formula(Calories_std ~ Steps_std + Time_std), df_std)
println("\nStandardized Coefficients (Reduced Model):")
println(coeftable(model_std3))
model_std7 = lm(@formula(Calories_std ~ Steps_std + Intensity_std), df_std)
println("\nStandardized Coefficients (Steps + Intensity):")
println(coeftable(model_std7))
CSV.write(joinpath(output_dir, "feature_importance.csv"),
    DataFrame(Predictor = ["Steps","Time"], Coef = coef(model_std3)[2:end],
             Abs = abs.(coef(model_std3)[2:end])))

# ============================================================
# PART 12: PREDICTION & VALIDATION
# ============================================================

println("\n" * "="^80)
println("PREDICTION VALIDATION")
println("="^80)

n = nrow(df_model)
train_size = floor(Int, 0.8 * n)
df_train = df_model[1:train_size, :]
df_test = df_model[train_size+1:end, :]
println("Training set: ", nrow(df_train), " observations")
println("Test set:     ", nrow(df_test), " observations")

model_train3 = lm(@formula(Calories ~ Steps + Time), df_train)
model_train7 = lm(@formula(Calories ~ Steps + Intensity), df_train)

function test_metrics(model, df_test)
    y_pred = predict(model, df_test)
    y_act = df_test.Calories
    rmse = sqrt(mean((y_act .- y_pred).^2))
    mae = mean(abs.(y_act .- y_pred))
    mape = mean(abs.((y_act .- y_pred) ./ y_act)) * 100
    return (rmse=rmse, mae=mae, mape=mape)
end

res3 = test_metrics(model_train3, df_test)
res7 = test_metrics(model_train7, df_test)

println("\nTest Set - Reduced Model (Steps+Time):")
println("  RMSE: $(round(res3.rmse, digits=2)), MAE: $(round(res3.mae, digits=2)), MAPE: $(round(res3.mape, digits=2))%")
println("\nTest Set - Steps+Intensity:")
println("  RMSE: $(round(res7.rmse, digits=2)), MAE: $(round(res7.mae, digits=2)), MAPE: $(round(res7.mape, digits=2))%")

p_pred = scatter(df_test.Calories, predict(model_train7, df_test),
    xlabel="Actual Calories", ylabel="Predicted Calories",
    title="Prediction Accuracy - Steps+Intensity (Test Set)", color=:steelblue, markersize=4, legend=false)
minv = min(minimum(df_test.Calories), minimum(predict(model_train7, df_test)))
maxv = max(maximum(df_test.Calories), maximum(predict(model_train7, df_test)))
plot!(p_pred, [minv, maxv], [minv, maxv], color=:red, linewidth=2, linestyle=:dash, label="Perfect Prediction")
savefig(p_pred, joinpath(output_dir, "prediction_accuracy.png"))
display(p_pred)
println("Saved: prediction_accuracy.png")

# ============================================================
# PART 13: FINAL SUMMARY REPORT
# ============================================================

report = """
================================================================================
STATISTICAL MODELING REPORT - CALORIES PREDICTION
================================================================================

DATASET:
  Source: $(input_path)
  Total observations: $(nrow(df_model))
  Date range: $(minimum(df_model.Date)) to $(maximum(df_model.Date))
  Variables: Steps, Distance, Time, Intensity (=Steps/Time) → Calories (response)

MULTICOLLINEARITY FINDINGS (Original Set):
  - Severe VIF for all three raw predictors (Steps, Distance, Time)
  - Steps-Distance correlation: $(round(corr_matrix[1,3], digits=3))

MULTICOLLINEARITY AFTER DERIVED FEATURE (Steps, Time, Intensity):
  - Intensity partially decorrelates the pace information from raw volume.
  - VIFs: Steps=$(round(vif_red[1], digits=1)), Time=$(round(vif_red[2], digits=1)), Intensity=$(round(vif_red[3], digits=1))

NEW MODELS WITH INTENSITY:
  Model 7: Calories ~ Steps + Intensity      (volume + pace)
  Model 8: Calories ~ Time + Intensity        (duration + pace)
  Model 9: Calories ~ Intensity               (pure pace)

MODEL COMPARISON:
$(join(map(row -> "  $(row.Model): R²=$(round(row.R2, digits=4)), RMSE=$(round(row.RMSE, digits=2))", eachrow(comparison)), "\n"))

RECOMMENDED MODEL:
  - Reduced Model (Steps + Time) remains mathematically clean and interpretable.
  - Steps + Intensity (Model 7) offers a physiological interpretation (total steps +
    average step rate) while also avoiding severe collinearity.
  Both perform similarly; choose based on whether “pace” is a more meaningful metric
  for your analysis.

TEST SET PERFORMANCE:
  Reduced (Steps+Time):  RMSE=$(round(res3.rmse, digits=2)), MAPE=$(round(res3.mape, digits=2))%
  Steps+Intensity:       RMSE=$(round(res7.rmse, digits=2)), MAPE=$(round(res7.mape, digits=2))%

================================================================================
"""

open(joinpath(output_dir, "modeling_report.txt"), "w") do f
    write(f, report)
end
println(report)
println("\nSaved: modeling_report.txt")

# ============================================================
# PART 14: SAVE MODEL COEFFICIENTS
# ============================================================

coef_df = DataFrame(
    Model = String[],
    Term = String[],
    Estimate = Float64[],
    StdError = Float64[],
    t_value = Float64[],
    p_value = Float64[]
)

for (mname, m) in [("Linear_Full", model1), ("Log-Log", model2), ("Reduced", model3),
                   ("PCA-$(ncomp)", model5),
                   ("Steps+Intensity", model7), ("Time+Intensity", model8), ("Intensity", model9)]
    terms = coefnames(m)
    ests = coef(m)
    ses = stderror(m)
    tvals = ests ./ ses
    df_res = dof_residual(m)
    pvals = 2 .* ccdf.(Ref(TDist(df_res)), abs.(tvals))
    for i in 1:length(terms)
        push!(coef_df, (mname, terms[i], ests[i], ses[i], tvals[i], pvals[i]))
    end
end

if model6 !== nothing
    m = model6
    terms = coefnames(m)
    ests = coef(m)
    ses = stderror(m)
    tvals = ests ./ ses
    df_res = dof_residual(m)
    pvals = 2 .* ccdf.(Ref(TDist(df_res)), abs.(tvals))
    for i in 1:length(terms)
        push!(coef_df, ("PCA-2", terms[i], ests[i], ses[i], tvals[i], pvals[i]))
    end
end

# Ridge
push!(coef_df, ("Ridge", "Intercept", ridge_result.intercept, NaN, NaN, NaN))
for (name, coef) in zip(predictor_names_orig, ridge_result.beta)
    push!(coef_df, ("Ridge", name, coef, NaN, NaN, NaN))
end

CSV.write(joinpath(output_dir, "model_coefficients.csv"), coef_df)
println("Saved: model_coefficients.csv")

println("\n" * "="^80)
println("COMPLETE! All outputs saved to: ", output_dir)
println("="^80)

# Interpretation of Modeling Results for Calories Prediction
## Интерпретация на резултатите от моделирането за прогнозиране на калории

This document interprets the outputs from the statistical modeling script. The analysis aimed to predict daily **Calories** from activity metrics (`Steps`, `Distance`, `Time`) while addressing severe multicollinearity through feature engineering (`Intensity = Steps / Time`), Ridge regression, and PCA.

Този документ интерпретира резултатите от скрипта за статистическо моделиране. Анализът имаше за цел да прогнозира дневните **калории** от метриките за активност (`Steps`, `Distance`, `Time`), като се справя със силна мултиколинеарност чрез инженеринг на характеристики (`Intensity = Steps / Time`), Ridge регресия и PCA.

---

## 1. Dataset Overview
## 1. Общ преглед на данните

| Metric | Value |
|--------|-------|
| **Observations** | 3033 complete days (all metrics > 0) |
| **Date range** | 2018-01-01 to 2026-05-31 |
| **Response variable** | Calories |
| **Predictors** | Steps, Distance, Time, Intensity (derived) |

**Наблюдения** | 3033 пълни дни (всички метрики > 0) |
**Период** | 2018-01-01 до 2026-05-31 |
**Целева променлива** | Калории |
**Предиктори** | Стъпки, Разстояние, Време, Интензивност (производна) |

---

## 2. Correlation Analysis: The Multicollinearity Problem
## 2. Корелационен анализ: Проблемът с мултиколинеарността

From `correlation_heatmap.png` and `correlation_matrix.csv`:

| | Steps | Calories | Distance | Time | Intensity |
|---|-------|----------|----------|------|-----------|
| **Steps** | 1.000 | 0.993 | 0.993 | 0.982 | 0.016 |
| **Calories** | 0.993 | 1.000 | 0.988 | 0.984 | 0.010 |
| **Distance** | 0.993 | 0.988 | 1.000 | 0.978 | 0.015 |
| **Time** | 0.982 | 0.984 | 0.978 | 1.000 | -0.018 |
| **Intensity** | 0.016 | 0.010 | 0.015 | -0.018 | 1.000 |

**Key observations / Ключови наблюдения:**

- **Calories is extremely strongly correlated** with Steps (0.993), Distance (0.988), and Time (0.984). This explains why models can achieve very high R² values.

- **Steps, Distance, and Time are also highly correlated** with each other (0.978–0.993). This is **severe multicollinearity** – they essentially measure the same underlying construct (total activity volume). Including all three in a single model causes unstable coefficient estimates.

- **Intensity** (Steps/Time) is **almost uncorrelated** with all other predictors (r ≈ 0.01–0.02). This is excellent – it captures a completely different aspect of physical activity: **pace**, rather than volume.

**Калориите са изключително силно корелирани** със Стъпки (0.993), Разстояние (0.988) и Време (0.984). Това обяснява защо моделите могат да постигнат много високи R² стойности.

**Стъпките, Разстоянието и Времето са силно корелирани** помежду си (0.978–0.993). Това е **тежка мултиколинеарност** – те измерват една и съща основна конструкция (общ обем на активността). Включването и на трите в един модел води до нестабилни оценки на коефициентите.

**Интензивността** (Стъпки/Време) е **почти некорелирана** с всички останали предиктори (r ≈ 0.01–0.02). Това е отлично – тя улавя напълно различен аспект на физическата активност: **темпото**, а не обема.

---

## 3. Normality Assessment: QQ Plots and Histograms
## 3. Оценка на нормалност: QQ графики и хистограми

- All variables (Steps, Calories, Distance, Time, Intensity) are **right‑skewed** – most values are low, with a long tail of high values.
- The **Jarque‑Bera tests** reject normality (p < 0.05) for all original variables.
- **Log‑transformed** variables show much better normality, especially `log(Calories)`, `log(Steps)`, and `log(Distance)`. This supports the use of a **Log‑Log model**.

Всички променливи (Стъпки, Калории, Разстояние, Време, Интензивност) са **с дясна асиметрия** – повечето стойности са ниски, с дълга опашка от високи стойности.

**Тестовете на Жарк‑Бера** отхвърлят нормалността (p < 0.05) за всички оригинални променливи.

**Логаритмуваните** променливи показват много по-добра нормалност, особено `log(Calories)`, `log(Steps)` и `log(Distance)`. Това подкрепя използването на **лог‑лог модел**.

---

## 4. Model Comparison Results
## 4. Резултати от сравнението на моделите

From `model_comparison.csv` and individual model summaries:

| Model | R² | Adj R² | RMSE | MAE | AIC |
|-------|-----|--------|------|-----|-----|
| **Log‑Log** | **0.9915** | **0.9915** | **0.078** | **0.044** | **-6848** |
| Linear OLS (full) | 0.9885 | 0.9885 | 28.94 | 16.89 | 20422 |
| Ridge (λ=1) | 0.9885 | 0.9885 | 28.94 | 16.90 | — |
| Reduced (Steps+Time) | 0.9883 | 0.9883 | 29.12 | 16.75 | 20457 |
| Steps+Intensity | 0.9861 | 0.9861 | 31.78 | 19.03 | 20988 |
| PCA-1 | 0.9861 | 0.9861 | 31.82 | 19.08 | 20994 |
| Time+Intensity | 0.9689 | 0.9689 | 47.58 | 27.20 | 23435 |
| Intensity Only | 0.0001 | -0.0002 | 269.65 | 202.40 | 33956 |

**Key insights / Ключови изводи:**

### Best Overall Model: Log‑Log
- **R² = 0.9915** – explains 99.15% of the variance in log(Calories).
- **Lowest AIC (-6848)** – significantly better than all other models.
- Interpreting log‑log coefficients: a 1% increase in Steps, Distance, or Time leads to a **0.43%, 0.36%, or 0.20%** increase in Calories respectively.
- **Limitation:** The response is on the log scale, so predictions need back‑transformation.

### Best Interpretable Model: Reduced (Steps + Time)
- **R² = 0.9883** – almost as good as the full OLS model.
- **Simple formula:** `Calories ≈ 0.456 + 0.032·Steps + 1.104·Time`.
- **Interpretation:** Each additional step adds ~0.032 calories; each extra minute of activity adds ~1.10 calories.
- **Advantage:** No multicollinearity issues (only Steps and Time, which still have moderate correlation but are manageable).

### Best Physiologically Interpretable Model: Steps + Intensity
- **R² = 0.9861** – only slightly lower than Reduced.
- **Formula:** `Calories ≈ 3.38 + 0.0426·Steps - 0.0126·Intensity`.
- This separates **volume** (total steps) from **pace** (steps per minute). Interestingly, Intensity has a **negative** coefficient – when holding Steps constant, higher intensity (faster pace) is associated with slightly **lower** calories. This could be because faster walking may cover the same distance in less time, reducing total duration.

### Simple / Physiologically Naïve: Time + Intensity
- **R² = 0.9689** – still quite good, but noticeably worse than the best models.
- Suggests that knowing total steps (or distance) is more important than just time and pace separately.

### Worst: Intensity Only
- **R² ≈ 0.0001** – essentially no predictive power.
- Intensity alone (steps per minute) explains almost nothing about total daily calories. This makes sense – calorie burn depends on total activity volume, not just pace.

---

## 5. Feature Importance (Reduced Model)
## 5. Значимост на характеристиките (редуциран модел)

From `feature_importance.csv` (standardized coefficients):

| Predictor | Standardized Coefficient | Importance |
|-----------|--------------------------|------------|
| **Steps** | **0.747** | Most important |
| **Time** | **0.250** | Secondary |

- **Steps** is about **3× more important** than Time in predicting calories.
- The interpretation: a 1‑standard‑deviation increase in Steps leads to a 0.747‑standard‑deviation increase in Calories, while the same for Time leads to only 0.250.

**Стъпките** са около **3 пъти по-важни** от Времето за прогнозиране на калориите.  
Тълкуване: увеличение с 1 стандартно отклонение в Стъпките води до 0.747 стандартни отклонения увеличение в Калориите, докато същото за Времето води до едва 0.250.

---

## 6. Residual Diagnostics (Reduced Model)
## 6. Диагностика на остатъците (редуциран модел)

From `residual_diagnostics.png`:

- **Residuals vs Fitted:** No clear pattern – residuals are randomly scattered around zero. This suggests a good linear fit.
- **Normal Q‑Q plot:** Residuals follow the normal line closely, indicating normality.
- **Histogram:** Residuals are approximately bell‑shaped.
- **Scale‑Location plot:** No obvious trend – variance is relatively constant.
- **Jarque‑Bera p‑value** for residuals > 0.05 – we **fail to reject normality**, meaning residuals are normally distributed.

**Остатъците са нормално разпределени** и не показват системни грешки – моделът е статистически валиден.

---

## 7. Test Set Validation
## 7. Валидация с тестов набор

From `modeling_report.txt`:

| Model | RMSE | MAPE |
|-------|------|------|
| Reduced (Steps+Time) | 28.94 | 3.66% |
| Steps+Intensity | 28.97 | 3.58% |

- Both models have **very low MAPE (< 4%)** – predictions are within ~4% of actual calories on average.
- This confirms that the models generalise well to unseen data.
- The `prediction_accuracy.png` plot shows points closely following the diagonal line (actual = predicted), with no major outliers.

И двата модела имат **много ниска MAPE (< 4%)** – прогнозите са средно в рамките на ~4% от действителните калории.  
Това потвърждава, че моделите се обобщават добре за нови данни.

---

## 8. Model Coefficients Comparison
## 8. Сравнение на коефициентите на моделите

From `model_coefficients.csv`:

| Model | Intercept | Steps | Distance | Time | Intensity |
|-------|-----------|-------|----------|------|-----------|
| Linear OLS (full) | 0.20 | 0.0280 | 6.185 | 1.063 | — |
| Reduced | 0.46 | **0.0320** | — | **1.104** | — |
| Steps+Intensity | 3.38 | **0.0426** | — | — | **-0.0126** |
| Time+Intensity | 1.06 | — | — | **4.346** | **0.0593** |

**Interpretation notes:**

- In the **full OLS model**, `Distance` has a very large coefficient (6.185) but this is due to multicollinearity – it doesn't mean 1 km of distance burns 6.185 calories while 1 step burns only 0.028. The units are different: Distance is in km, Steps in thousands (the data scale matters).

- The **Reduced model** gives more stable estimates: **0.032 calories per step** and **1.104 calories per minute** of activity.

- In the **Steps+Intensity model**, the negative coefficient for Intensity suggests that when you control for total steps, faster pace is actually associated with lower calorie burn. This is plausible because faster walking may mean less time spent active.

**В пълния OLS модел** `Distance` има много голям коефициент (6.185), но това се дължи на мултиколинеарност – това не означава, че 1 км изминато разстояние изгаря 6.185 калории, докато 1 стъпка изгаря само 0.028. Мерните единици са различни.

**Редуцираният модел** дава по-стабилни оценки: **0.032 калории на стъпка** и **1.104 калории на минута** активност.

---

## 9. Recommendations and Practical Use
## 9. Препоръки и практическа употреба

### 🏆 Recommended Model:
**Reduced Model: Calories ~ Steps + Time**

- **Best balance** of accuracy, interpretability, and simplicity.
- Formula: `Calories ≈ 0.456 + 0.032·Steps + 1.104·Time`
- Works in any situation where you have step count and activity duration.
- No need for distance (which often requires GPS) – easier to track with a basic pedometer.

### 🔬 For Physiological Insight:
**Steps + Intensity Model**

- Separates **volume** (steps) from **pace** (steps/min).
- Useful if you want to understand whether someone is walking more steps, walking faster, or both.
- Slightly lower R² (0.986 vs 0.988) – a small trade‑off for richer interpretation.

### ⚠️ Avoid:
- **Full OLS** (multicollinearity makes coefficients unstable).
- **Intensity Only** (essentially no predictive power).
- **Time + Intensity** (lower accuracy, less intuitive).

---

## 10. Summary Table
## 10. Обобщаваща таблица

| Metric | Finding |
|--------|---------|
| **Best performing model** | Log‑Log (R²=0.9915) |
| **Best interpretable model** | Reduced (Steps+Time) (R²=0.9883) |
| **Most physiologically meaningful** | Steps+Intensity (R²=0.9861) |
| **Most important predictor** | Steps (standardised coefficient 0.747) |
| **Prediction accuracy (MAPE)** | ~3.6% – excellent! |
| **Residuals** | Normal, homoscedastic – model assumptions satisfied |
| **Multicollinearity resolved** | Yes – Reduced and Intensity models avoid severe VIF |

---

**All analysis outputs are saved in `./modeling/`. The chosen model is ready for deployment or further refinement.**

**Всички резултати от анализа са записани в `./modeling/`. Избраният модел е готов за внедряване или допълнително усъвършенстване.**

# Multi-Metric Time Series Modeling Pipeline
## Многомерен тръбопровод за моделиране на времеви редове

This pipeline performs comprehensive time series analysis and forecasting for four key metrics: **Steps, Calories, Distance, and Time**.  
Този тръбопровод извършва задълбочен анализ на времеви редове и прогнозиране за четири ключови метрики: **Стъпки, Калории, Разстояние и Време**.

It builds and compares models (ARMA, ARIMA, SARIMA, SARIMAX, GARCH), checks stationarity, analyzes autocorrelation and seasonality, generates diagnostic plots, and produces 30‑day out‑of‑sample forecasts—all saved locally and displayed interactively.  
Той изгражда и сравнява модели (ARMA, ARIMA, SARIMA, SARIMAX, GARCH), проверява стационарността, анализира автокорелацията и сезонността, генерира диагностични графики и произвежда 30‑дневни прогнози извън извадката – всичко това се записва локално и се показва интерактивно на екрана.

---

## 1. Overview of the Pipeline Structure
## 1. Структура на тръбопровода

The code is organised into modular sections, making it easy to adapt or extend.  
Кодът е организиран в модулни секции, което го прави лесен за адаптиране или разширяване.

The main components are:  
Основните компоненти са:

1. **Setup** – load packages, configure output directories, set global plot defaults.  
   **Настройка** – зареждане на пакети, конфигуриране на изходни директории, задаване на глобални настройки за графики.

2. **Helper functions** – date parsing, stationarity tests (ADF), autocorrelation (ACF/PACF), Ljung‑Box test, safe model extraction (AIC, BIC, residuals, forecasts).  
   **Помощни функции** – разпознаване на дати, тестове за стационарност (ADF), автокорелация (ACF/PACF), тест на Люнг‑Бокс, безопасно извличане на модели (AIC, BIC, остатъци, прогнози).

3. **Core pipeline** – a single function `execute_metric_pipeline` that processes one metric at a time.  
   **Основен тръбопровод** – една функция `execute_metric_pipeline`, която обработва по една метрика.

4. **Main loop** – loads the CSV, parses dates, and runs the pipeline for all four metrics.  
   **Основен цикъл** – зарежда CSV, разпознава дати и изпълнява тръбопровода за всичките четири метрики.

The pipeline is **fully automated** – for each metric it performs:  
Тръбопроводът е **напълно автоматизиран** – за всяка метрика той изпълнява:

- Data cleaning and min‑max scaling.  
  Почистване на данни и мащабиране по минимума и максимума.

- Descriptive statistics and visualisation with trend and moving average.  
  Дескриптивна статистика и визуализация с тенденция и пълзяща средна.

- Stationarity assessment (ADF test, differencing).  
  Оценка на стационарност (ADF тест, диференциране).

- ACF/PACF plots for order selection.  
  ACF/PACF графики за избор на порядък.

- Seasonal profiles (monthly and weekly patterns).  
  Сезонни профили (месечни и седмични модели).

- Fitting of multiple ARIMA/ARMA models, selection of the best by AIC.  
  Напасване на множество ARIMA/ARMA модели, избор на най-добрия по AIC.

- SARIMAX with exogenous regressors (day‑of‑week, month).  
  SARIMAX с екзогенни регресори (ден от седмицата, месец).

- GARCH volatility modelling on residuals.  
  GARCH моделиране на волатилността на остатъците.

- Residual diagnostics (QQ plots, histograms, ACF).  
  Диагностика на остатъците (QQ графики, хистограми, ACF).

- 30‑day forecast with confidence intervals.  
  30‑дневна прогноза с доверителни интервали.

All plots are both saved to `./time_series_modelling/plots/` and displayed on screen for immediate inspection.  
Всички графики се записват в `./time_series_modelling/plots/` и се показват на екрана за незабавна проверка.

---

## 2. Detailed Explanation of Each Section
## 2. Подробно обяснение на всяка част

### 2.1 Setup and Directories
### 2.1 Настройка и директории

- Packages: `CSV`, `DataFrames`, `Dates`, `Statistics`, `LinearAlgebra`, `Plots`, `StatsPlots`, `Distributions`, `Printf`, `Random`, `StatsBase`, `ARCHModels`, `StateSpaceModels`.  
  Пакети: `CSV`, `DataFrames`, `Dates`, `Statistics`, `LinearAlgebra`, `Plots`, `StatsPlots`, `Distributions`, `Printf`, `Random`, `StatsBase`, `ARCHModels`, `StateSpaceModels`.

- Output folders are created under `time_series_modelling/`:  
  Изходните папки се създават под `time_series_modelling/`:
  - `plots/` – all visualisations.  
    `plots/` – всички визуализации.
  - `results/` – model summaries and diagnostics.  
    `results/` – обобщения на модели и диагностика.
  - `forecasts/` – CSV files with 30‑day predictions.  
    `forecasts/` – CSV файлове с 30‑дневни прогнози.

### 2.2 Safe Interface Helpers
### 2.2 Помощници за безопасен интерфейс

Because different packages have different APIs (e.g., `StateSpaceModels` vs `ARCHModels`), the code provides wrapper functions to extract AIC, BIC, log‑likelihood, residuals, and forecasts regardless of the underlying model type. This ensures robustness when switching between model families.  
Тъй като различните пакети имат различни API (напр. `StateSpaceModels` срещу `ARCHModels`), кодът предоставя обвиващи функции за извличане на AIC, BIC, логаритмична правдоподобност, остатъци и прогнози, независимо от типа на модела. Това гарантира устойчивост при превключване между семейства модели.

### 2.3 Data Parsing and Math Helpers
### 2.3 Разпознаване на данни и математически помощници

- **`parse_date_string`** – handles multiple date formats (dot‑separated, ISO, etc.) and converts to `Date`.  
  **`parse_date_string`** – обработва множество формати на дати (с точки, ISO и др.) и преобразува в `Date`.

- **`adf_test`** – Augmented Dickey‑Fuller test for stationarity (with optional lags).  
  **`adf_test`** – разширен тест на Дики‑Фулър за стационарност (с опционални изоставания).

- **`compute_acf` / `compute_pacf`** – manual calculations of autocorrelation and partial autocorrelation functions.  
  **`compute_acf` / `compute_pacf`** – ръчни изчисления на автокорелационната и частичната автокорелационна функция.

- **`ljung_box`** – Ljung‑Box test for white noise.  
  **`ljung_box`** – тест на Люнг‑Бокс за бял шум.

- **`find_column_symbol`** – safe column lookup in DataFrame by name.  
  **`find_column_symbol`** – безопасно търсене на колона в DataFrame по име.

### 2.4 Core Pipeline Function `execute_metric_pipeline`
### 2.4 Основна функция на тръбопровода `execute_metric_pipeline`

This is the heart of the code. It processes one metric (e.g., `"Steps"`) and performs the following steps:  
Това е сърцето на кода. Той обработва една метрика (например `"Steps"`) и изпълнява следните стъпки:

#### A. Data Preparation
#### A. Подготовка на данните

- Drops missing values for the target column and `Date`.  
  Премахва липсващите стойности за целевата колона и `Date`.

- Sorts by date.  
  Сортира по дата.

- Applies **min‑max scaling** to stabilise numerical convergence (especially for GARCH and ARIMA fitting).  
  Прилага **мащабиране по минимум‑максимум** за стабилизиране на числовата конвергенция (особено за GARCH и ARIMA).

- Prints summary statistics (mean, std, min, max).  
  Отпечатва обобщена статистика (средна, стандартно отклонение, минимум, максимум).

#### B. Time Series Visualisation (Plot 1)
#### B. Визуализация на времевия ред (Графика 1)

- Plots the raw series with a **30‑day rolling average** (red) and a **linear trend line** (green dashed).  
  Изобразява суровия ред с **30‑дневна пълзяща средна** (червено) и **линейна тенденция** (зелено пунктирано).

- Shows the overall trend and short‑term fluctuations.  
  Показва общата тенденция и краткосрочните колебания.

- Saved as `01_time_series_{metric}.png`.  
  Записва се като `01_time_series_{metric}.png`.

#### C. Stationarity Analysis (Plot 2)
#### C. Анализ на стационарността (Графика 2)

- Computes the **ADF test** statistic for the original series and for the **first‑differenced** series.  
  Изчислява статистиката на **ADF теста** за оригиналния ред и за **първо‑диференцирания** ред.

- Plots both the original level and the differenced series to visually assess stationarity.  
  Изобразява както оригиналното ниво, така и диференцирания ред за визуална оценка на стационарността.

- Saved as `02_stationarity_{metric}.png`.  
  Записва се като `02_stationarity_{metric}.png`.

#### D. Autocorrelation Analysis (Plot 3)
#### D. Анализ на автокорелацията (Графика 3)

- Computes and plots **ACF** and **PACF** up to a maximum lag (≤ 50).  
  Изчислява и изобразява **ACF** и **PACF** до максимално изоставане (≤ 50).

- Includes 95% confidence intervals (red dashed lines) to identify significant lags.  
  Включва 95% доверителни интервали (червени пунктирани линии) за идентифициране на значими изоставания.

- Helps determine the orders (p, d, q) for ARIMA.  
  Помага за определяне на порядъците (p, d, q) за ARIMA.

- Saved as `03_acf_pacf_{metric}.png`.  
  Записва се като `03_acf_pacf_{metric}.png`.

#### E. Seasonality Profiles (Plot 4)
#### E. Сезонни профили (Графика 4)

- Groups data by **month** and **day‑of‑week**.  
  Групира данните по **месец** и **ден от седмицата**.

- Creates two bar charts showing the average value of the metric for each month and each weekday.  
  Създава две стълбови диаграми, показващи средната стойност на метриката за всеки месец и за всеки ден от седмицата.

- Reveals weekly and annual patterns (e.g., higher steps on weekends, lower in winter).  
  Разкрива седмични и годишни модели (например повече стъпки през уикенда, по-малко през зимата).

- Saved as `04_seasonality_{metric}.png`.  
  Записва се като `04_seasonality_{metric}.png`.

#### F. Model Fitting (ARIMA / ARMA)
#### F. Напасване на модели (ARIMA / ARMA)

- Tests a set of candidate ARIMA orders: `(1,0,0)`, `(0,0,1)`, `(1,0,1)`, `(1,1,1)`, `(2,1,1)`.  
  Тества набор от кандидат-порядъци на ARIMA: `(1,0,0)`, `(0,0,1)`, `(1,0,1)`, `(1,1,1)`, `(2,1,1)`.

- Uses `SARIMA` from `StateSpaceModels` (with optional seasonality) and falls back to `ARMA` from `ARCHModels` if needed.  
  Използва `SARIMA` от `StateSpaceModels` (с опционална сезонност) и се връща към `ARMA` от `ARCHModels` при необходимост.

- For each fitted model, extracts AIC and BIC.  
  За всеки напаснат модел извлича AIC и BIC.

- Selects the model with the **lowest AIC** as the best.  
  Избира модела с **най-нисък AIC** за най-добър.

- Writes a summary file `results/model_summary_{metric}.txt` listing all fitted models and their metrics.  
  Записва обобщаващ файл `results/model_summary_{metric}.txt`, съдържащ всички напаснати модели и техните метрики.

#### G. SARIMAX with Exogenous Regressors
#### G. SARIMAX с екзогенни регресори

- Adds four exogenous variables: `sin(2π·DayOfWeek/7)`, `cos(2π·DayOfWeek/7)`, `sin(2π·Month/12)`, `cos(2π·Month/12)`.  
  Добавя четири екзогенни променливи: `sin(2π·DayOfWeek/7)`, `cos(2π·DayOfWeek/7)`, `sin(2π·Month/12)`, `cos(2π·Month/12)`.

- Fits a `SARIMAX(1,0,0)(1,0,0)[7]` model (seasonal period 7 days).  
  Напасва модел `SARIMAX(1,0,0)(1,0,0)[7]` (сезонен период 7 дни).

- Includes its AIC in the summary file.  
  Включва неговия AIC в обобщаващия файл.

#### H. GARCH Residual Volatility Modelling
#### H. GARCH моделиране на волатилността на остатъците

- Extracts residuals from the best‑fitting ARIMA model.  
  Извлича остатъците от най‑добрия ARIMA модел.

- Fits a **GARCH(1,1)** model to the residuals to capture conditional heteroscedasticity (volatility clustering).  
  Напасва **GARCH(1,1)** модел към остатъците, за да улови условна хетероскедастичност (групиране на волатилността).

- Saves the GARCH output to `results/garch_diagnostics_{metric}.txt`.  
  Записва резултата от GARCH в `results/garch_diagnostics_{metric}.txt`.

#### I. Residual Diagnostics (Plot 5)
#### I. Диагностика на остатъците (Графика 5)

- For the best ARIMA model, produces a **4‑panel diagnostic plot**:  
  За най‑добрия ARIMA модел създава **4‑панелна диагностична графика**:
  1. Residuals over time (check for patterns).  
     Остатъци във времето (проверка за модели).
  2. Normal Q‑Q plot (check for normality).  
     Нормална Q‑Q графика (проверка за нормалност).
  3. Histogram with density (visual distribution).  
     Хистограма с плътност (визуално разпределение).
  4. ACF of residuals (check for white noise).  
     ACF на остатъците (проверка за бял шум).

- Saved as `05_diagnostics_{metric}.png`.  
  Записва се като `05_diagnostics_{metric}.png`.

#### J. Out‑of‑Sample Forecasting (Plot 6 and CSV)
#### J. Прогнозиране извън извадката (Графика 6 и CSV)

- Generates a **30‑day forecast** using the best model.  
  Генерира **30‑дневна прогноза** чрез най‑добрия модел.

- Rescales the forecast back to original units using the min‑max scaling parameters.  
  Премащабира прогнозата обратно към оригиналните единици, използвайки параметрите на мащабирането.

- Computes 95% prediction intervals (assuming normality).  
  Изчислява 95% прогнозни интервали (при допускане на нормалност).

- Saves the forecast as `forecasts/forecast_out_30d_{metric}.csv`.  
  Записва прогнозата като `forecasts/forecast_out_30d_{metric}.csv`.

- Plots the last 120 days of historical data alongside the forecast with uncertainty bands (ribbon).  
  Изобразява последните 120 дни от историческите данни заедно с прогнозата и интервали на неопределеност (лента).

- Saved as `06_forecast_{metric}.png`.  
  Записва се като `06_forecast_{metric}.png`.

### 2.5 Main Execution
### 2.5 Основно изпълнение

- Loads `Steps_combined.csv` and parses the `Day` column into `Date`.  
  Зарежда `Steps_combined.csv` и разпознава колоната `Day` като `Date`.

- Loops over the four metrics (`"Steps"`, `"Time"`, `"Distance"`, `"Calories"`).  
  Обхожда четирите метрики (`"Steps"`, `"Time"`, `"Distance"`, `"Calories"`).

- For each, calls `execute_metric_pipeline`.  
  За всяка извиква `execute_metric_pipeline`.

- Prints a final completion message.  
  Отпечатва финално съобщение за завършване.

---

## 3. Key Design Decisions
## 3. Основни проектни решения

- **Min‑max scaling** – ensures that different metrics (with vastly different scales) are treated equally during model fitting, and it also stabilises the optimisation of GARCH and ARIMA.  
  **Мащабиране по минимум‑максимум** – гарантира, че различните метрики (с много различни мащаби) се третират еднакво по време на напасване на моделите и също така стабилизира оптимизацията на GARCH и ARIMA.

- **Robust model interfaces** – the wrapper functions allow seamless switching between `StateSpaceModels` and `ARCHModels` without changing the core logic.  
  **Устойчиви интерфейси на моделите** – обвиващите функции позволяват безпроблемно превключване между `StateSpaceModels` и `ARCHModels` без промяна на основната логика.

- **Interactive display** – every plot is both saved and shown (`display(p)`) so that the user can immediately inspect results in Jupyter Lab.  
  **Интерактивно показване** – всяка графика се записва и показва (`display(p)`), така че потребителят веднага да може да разгледа резултатите в Jupyter Lab.

- **Automatic model selection** – the best ARIMA/ARMA model is chosen automatically by AIC, saving manual effort.  
  **Автоматичен избор на модел** – най‑добрият ARIMA/ARMA модел се избира автоматично по AIC, спестявайки ръчен труд.

- **Exogenous variables** – SARIMAX captures weekly and annual seasonality more explicitly than simple ARIMA, potentially improving forecast accuracy.  
  **Екзогенни променливи** – SARIMAX улавя седмичната и годишна сезонност по-изрично от обикновения ARIMA, потенциално подобрявайки точността на прогнозите.

- **Comprehensive diagnostics** – ADF, ACF/PACF, Ljung‑Box, residual QQ plots, and GARCH volatility checks ensure the models are statistically sound.  
  **Изчерпателна диагностика** – ADF, ACF/PACF, тест на Люнг‑Бокс, QQ графики на остатъците и проверки на волатилността с GARCH гарантират, че моделите са статистически обосновани.

---

## 4. Output Directory Structure and Files
## 4. Структура на изходните директории и файлове

After running the pipeline, the folder `time_series_modelling/` will contain:  
След изпълнение на тръбопровода, папката `time_series_modelling/` ще съдържа:


In [ ]:
# =============================================================================
# TIME SERIES MODELING PIPELINE FOR MULTI-TRACKING METRICS
# (Interactive version – plots are both saved and displayed on screen)
# =============================================================================
# Target Metrics: Steps, Calories, Distance, Time
# Models: ARMA, ARIMA, SARIMA, SARIMAX, GARCH
# =============================================================================

using CSV, DataFrames, Dates, Statistics, LinearAlgebra
using Plots, StatsPlots, Distributions, Printf, Random, StatsBase
using ARCHModels
using StateSpaceModels

# Set plot backend and global visual defaults
gr()
default(size=(1200, 700), dpi=150, legend=:topright)

# =============================================================================
# OUTPUT DIRECTORY SETUP
# =============================================================================
base_dir = "time_series_modelling"
mkpath(base_dir)
mkpath(joinpath(base_dir, "plots"))
mkpath(joinpath(base_dir, "results"))
mkpath(joinpath(base_dir, "forecasts"))

println("="^80)
println("MULTI-METRIC TIME SERIES MODELING PIPELINE INITIALIZED")
println("Dataset: Steps_combined.csv")
println("All localized outputs will be saved to: $base_dir/")
println("="^80)

# =============================================================================
# SAFE INTERFACE HELPERS (Gracefully handles diverse structural model types)
# =============================================================================
function safe_aic(m)
    try return m.aic catch; end
    try return aic(m) catch; end
    return NaN
end

function safe_bic(m)
    try return m.bic catch; end
    try return bic(m) catch; end
    return NaN
end

function safe_loglik(m)
    try return m.loglik catch; end
    try return loglikelihood(m) catch; end
    return NaN
end

function safe_residuals(m)
    try return residuals(m) catch; end
    try return m.residuals catch; end
    return Float64[]
end

function safe_forecast(m, h)
    # Approach A: StateSpaceModels
    try
        fc_obj = forecast(m, h)
        return (forecast=fc_obj.forecast, variance=fc_obj.variance)
    catch
    end
    # Approach B: ARCHModels multi-step prediction
    try
        fc_mean = predict(m, :mean, nsteps=h)
        fc_vol = predict(m, :volatility, nsteps=h)
        return (forecast=fc_mean, variance=fc_vol.^2)
    catch
    end
    # Approach C: ARCHModels static value fallback
    try
        fc_mean_val = predict(m, :return)
        fc_var_val  = predict(m, :variance)
        return (forecast=fill(fc_mean_val, h), variance=fill(fc_var_val, h))
    catch
    end
    return nothing
end

# =============================================================================
# DATA PARSING & MATH HELPERS
# =============================================================================
function parse_date_string(dt_str)
    formats = ["d.m.yyyy", "yyyy-mm-dd", "d.m.yy"]
    for fmt in formats
        try return Date(dt_str, fmt) catch; continue; end
    end
    if occursin(r"^\d+\.\d+\.\d+$", dt_str)
        parts = split(dt_str, ".")
        if length(parts) == 3
            d, m, y = parse.(Int, parts)
            if y < 100; y += 2000; end
            return Date(y, m, d)
        end
    end
    return missing
end

function adf_test(y::Vector{Float64}; lags::Int=1)
    n = length(y)
    if n <= lags+2; return NaN, zeros(0), zeros(0); end
    dy = diff(y)
    X = ones(n-lags-1, lags+2)
    for i in 1:(n-lags-1)
        X[i, 2] = y[i+lags]
        for j in 1:lags; X[i, 2+j] = dy[i+lags-j]; end
    end
    Y = dy[(lags+1):end]
    β = X \ Y
    residuals = Y - X * β
    σ² = sum(residuals.^2) / (length(Y) - size(X, 2))
    var_β = σ² * inv(X'X)
    se_β1 = sqrt(var_β[2, 2])
    t_stat = β[2] / se_β1
    return t_stat, β, residuals
end

function compute_acf(x::Vector{Float64}, max_lags::Int)
    n = length(x)
    if n <= 1; return ones(max_lags+1); end
    x_centered = x .- mean(x)
    acf_vals = zeros(max_lags + 1)
    for k in 0:max_lags
        if k == 0
            acf_vals[k+1] = 1.0
        else
            c_k = sum(x_centered[1:(n-k)] .* x_centered[(k+1):n]) / n
            c_0 = sum(x_centered.^2) / n
            acf_vals[k+1] = c_k / c_0
        end
    end
    return acf_vals
end

function compute_pacf(x::Vector{Float64}, max_lags::Int)
    acf_vals = compute_acf(x, max_lags)
    pacf_vals = zeros(max_lags + 1)
    pacf_vals[1] = 1.0
    for k in 1:max_lags
        R = zeros(k, k)
        for i in 1:k, j in 1:k
            R[i, j] = acf_vals[abs(i-j) + 1]
        end
        r = acf_vals[2:(k+1)]
        φ = R \ r
        pacf_vals[k+1] = φ[end]
    end
    return pacf_vals
end

function ljung_box(resid, lags::Int)
    n = length(resid)
    if n <= lags; return NaN, NaN; end
    r = autocor(resid, 1:lags)
    Q = n*(n+2)*sum(r.^2 ./ (n .- (1:lags)))
    pval = 1 - cdf(Chisq(lags), Q)
    return Q, pval
end

function find_column_symbol(df, target_name)
    cols = string.(names(df))
    idx = findfirst(x -> lowercase(x) == lowercase(target_name), cols)
    return idx !== nothing ? Symbol(cols[idx]) : nothing
end

# =============================================================================
# ISOLATED PIPELINE EXECUTION ENGINE
# =============================================================================
function execute_metric_pipeline(metric_name, col_sym, df_source, base_dir)
    println("\n" * "="^80)
    println("STARTING PROCESSING FOR METRIC: $(uppercase(metric_name))")
    println("="^80)

    # 1. Clean data rows specifically for this target column slice
    df_clean = dropmissing(df_source, [col_sym, :Date])
    df_clean = sort(df_clean, :Date)
    
    if nrow(df_clean) < 40
        println("Skipping $metric_name: Insufficient observations ($(nrow(df_clean)) rows).")
        return
    end

    dates = df_clean.Date
    raw_series = Vector{Float64}(df_clean[:, col_sym])

    # Perform min-max scaling to stabilize convergence profiles
    series_min = minimum(raw_series)
    series_max = maximum(raw_series)
    if series_max == series_min; series_max += 1e-6; end 
    scaled_series = (raw_series .- series_min) ./ (series_max - series_min)

    # Print Descriptive Metrics
    println("\n--- Summary Statistics ---")
    println("Total Records: $(nrow(df_clean))")
    println("Mean Level:    $(round(mean(raw_series), digits=2))")
    println("Std Deviation: $(round(std(raw_series), digits=2))")
    println("Range:         $(round(series_min, digits=2)) to $(round(series_max, digits=2))")

    # 2. Section 2: Time Series Visualization with Trendline
    t_idx = 1:length(raw_series)
    trend_model = [ones(length(t_idx)) t_idx] \ raw_series
    trend_line = [ones(length(t_idx)) t_idx] * trend_model

    p1 = plot(dates, raw_series, 
        title="Daily $metric_name History",
        xlabel="Date", ylabel=metric_name,
        linewidth=0.8, color=:steelblue, alpha=0.7,
        label="Daily Data", xrotation=30, size=(1400, 500))
    
    window = 30
    rolling_mean = [mean(raw_series[max(1, i-window+1):i]) for i in 1:length(raw_series)]
    plot!(p1, dates, rolling_mean, linewidth=2, color=:red, label="30-day Moving Average")
    plot!(p1, dates, trend_line, linewidth=2, color=:green, linestyle=:dash, label="Linear Trend")
    savefig(p1, joinpath(base_dir, "plots/01_time_series_$(metric_name).png"))
    display(p1)   # 👈 Shows plot on screen

    # 3. Section 3: Stationarity Analysis (ADF Tests)
    println("\n--- Augmented Dickey-Fuller Test ---")
    adf_stat, _, _ = adf_test(scaled_series)
    println("Original Series ADF Statistic: $(round(adf_stat, digits=4))")
    
    series_diff = diff(scaled_series)
    dates_diff = dates[2:end]
    adf_stat_d, _, _ = adf_test(series_diff)
    println("Differenced Series ADF Statistic: $(round(adf_stat_d, digits=4))")

    p2 = plot(layout=(2,1), size=(1400, 800), xrotation=30)
    plot!(p2[1], dates, raw_series, title="Original $metric_name Level", ylabel=metric_name, color=:steelblue, label="")
    plot!(p2[2], dates_diff, series_diff, title="First Differenced (Scaled Variance)", ylabel="Δ $metric_name", color=:darkgreen, label="")
    savefig(p2, joinpath(base_dir, "plots/02_stationarity_$(metric_name).png"))
    display(p2)

    # 4. Section 4: Autocorrelation Analysis (ACF / PACF)
    max_lags = min(50, length(scaled_series) ÷ 4)
    acf_vals = compute_acf(scaled_series, max_lags)
    pacf_vals = compute_pacf(scaled_series, max_lags)
    lags = 0:max_lags
    ci_limit = 1.96 / sqrt(length(scaled_series))
    
    p3 = plot(layout=(2,1), size=(1400, 700))
    bar!(p3[1], lags, acf_vals, title="ACF - $metric_name", xlabel="Lags", ylabel="Autocorrelation", color=:steelblue, alpha=0.8, label="")
    hline!(p3[1], [ci_limit, -ci_limit], color=:red, linestyle=:dash, label="95% CI")
    bar!(p3[2], lags, pacf_vals, title="PACF - $metric_name", xlabel="Lags", ylabel="Partial Autocorrelation", color=:darkorange, alpha=0.8, label="")
    hline!(p3[2], [ci_limit, -ci_limit], color=:red, linestyle=:dash, label="95% CI")
    savefig(p3, joinpath(base_dir, "plots/03_acf_pacf_$(metric_name).png"))
    display(p3)

    # 5. Section 5: Seasonality Profiles
    df_clean.Month = month.(df_clean.Date)
    df_clean.DayOfWeek = dayofweek.(df_clean.Date)

    monthly_avg = combine(groupby(df_clean, :Month), col_sym => mean => :AvgVal)
    weekly_avg = combine(groupby(df_clean, :DayOfWeek), col_sym => mean => :AvgVal)

    p4 = plot(layout=(2,1), size=(1200, 750))
    
    weekday_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    bar!(p4[1], monthly_avg.Month, monthly_avg.AvgVal, title="Seasonal Profile: Average $metric_name by Month", 
         xlabel="Month", ylabel="Mean Value", color=:steelblue, alpha=0.85, xticks=(1:12, month_labels), label="")
    bar!(p4[2], weekly_avg.DayOfWeek, weekly_avg.AvgVal, title="Weekly Routine Profile: Average $metric_name by Day", 
         xlabel="Day of Week", ylabel="Mean Value", color=:darkorange, alpha=0.85, xticks=(1:7, weekday_labels), label="")
    savefig(p4, joinpath(base_dir, "plots/04_seasonality_$(metric_name).png"))
    display(p4)

    # 6. Section 6: Model Estimations (ARIMA / ARMA)
    println("\n--- Fitting Mathematical Spec Models ---")
    y = scaled_series
    
    function safe_sarima_fit(y; order, seasonal=(0,0,0,0), X=nothing)
        try
            if X !== nothing; return SARIMA(y; order=order, seasonal=seasonal, xreg=X)
            else; return SARIMA(y; order=order, seasonal=seasonal); end
        catch; return nothing; end
    end

    candidate_orders = [(1,0,0), (0,0,1), (1,0,1), (1,1,1), (2,1,1)]
    arima_models = Dict()
    for (p,d,q) in candidate_orders
        label = "ARIMA($p,$d,$q)"
        m_fit = safe_sarima_fit(y; order=(p,d,q))
        if m_fit !== nothing; arima_models[label] = m_fit; end
    end

    # Fallback to ARCHModels framework if native convergence fails
    if isempty(arima_models)
        for p in 1:2, q in 1:2
            try
                m_arma = fit(ARMA{p,q}, y)
                arima_models["ARMA($p,$q)"] = m_arma
            catch; end
        end
    end

    best_key = nothing
    best_model = nothing
    if !isempty(arima_models)
        best_key = argmin(Dict(k => safe_aic(arima_models[k]) for k in keys(arima_models)))
        best_model = arima_models[best_key]
        println("Identified Best Fit Structure: $best_key")
    else
        println("[Warning]: Primary linear structures failed to reach optimal criteria.")
    end

    # 7. Section 7: SARIMAX Exogenous Feature Injection
    X_reg = Matrix{Float64}(undef, length(y), 4)
    X_reg[:,1] = sin.(2π * df_clean.DayOfWeek ./ 7)
    X_reg[:,2] = cos.(2π * df_clean.DayOfWeek ./ 7)
    X_reg[:,3] = sin.(2π * df_clean.Month ./ 12)
    X_reg[:,4] = cos.(2π * df_clean.Month ./ 12)
    
    sarimax_models = Dict()
    try
        sx_fit = safe_sarima_fit(y; order=(1,0,0), seasonal=(1,0,0,7), X=X_reg)
        if sx_fit !== nothing; sarimax_models["SARIMAX(1,0,0)(1,0,0)[7]"] = sx_fit; end
    catch; end

    # Log Metrics Comparisons Tables
    open(joinpath(base_dir, "results/model_summary_$(metric_name).txt"), "w") do io
        write(io, "MODEL METRICS BREAKDOWN FOR TRACE: $metric_name\n" * "="^60 * "\n")
        for (k, m) in arima_models
            write(io, "Model: $k | AIC: $(round(safe_aic(m), digits=2)) | BIC: $(round(safe_bic(m), digits=2))\n")
        end
        for (k, m) in sarimax_models
            write(io, "Model: $k | AIC: $(round(safe_aic(m), digits=2))\n")
        end
    end

    # 8. Section 8: GARCH Volatility Residual Profile Check
    if best_model !== nothing
        resid = safe_residuals(best_model)
        if !isempty(resid)
            try
                garch_fit = fit(GARCH{1,1}, resid)
                open(joinpath(base_dir, "results/garch_diagnostics_$(metric_name).txt"), "w") do io
                    show(io, garch_fit)
                end
            catch; end
        end
    end

    # 9. Section 9: Structural Model Diagnostics
    if best_model !== nothing
        resid = safe_residuals(best_model)
        if length(resid) > 15
            p5 = plot(layout=(2,2), size=(1400, 900))
            plot!(p5[1], dates[1:length(resid)], resid, title="Model Innovations (Residuals)", color=:steelblue, xrotation=30, label="")
            qqplot!(p5, Normal(), resid, subplot=2, title="Quantile-Quantile Normal Plot", color=:steelblue, label="")
            histogram!(p5[3], resid, bins=40, title="Innovations Distribution Histogram", color=:steelblue, alpha=0.7, label="")
            acf_res = compute_acf(resid, min(20, length(resid)÷4))
            bar!(p5[4], 0:(length(acf_res)-1), acf_res, title="Residual ACF (White Noise Validation)", color=:darkgreen, label="")
            savefig(p5, joinpath(base_dir, "plots/05_diagnostics_$(metric_name).png"))
            display(p5)
        end
    end

    # 10. Section 10: Out-of-Sample Forecasting
    h = 30
    fc_dates = dates[end] .+ Day.(1:h)

    if best_model !== nothing
        fc_data = safe_forecast(best_model, h)
        if fc_data !== nothing
            try
                fc_m_scaled = fc_data.forecast
                fc_v_scaled = fc_data.variance

                # Re-invert Min-Max Scaling to output actual data units
                fc_mean_orig = series_min .+ (series_max - series_min) .* fc_m_scaled
                fc_std_orig  = (series_max - series_min) .* sqrt.(fc_v_scaled)
                fc_lower = fc_mean_orig .- 1.96 .* fc_std_orig
                fc_upper = fc_mean_orig .+ 1.96 .* fc_std_orig

                df_forecast_out = DataFrame(Date=fc_dates, Forecast=fc_mean_orig, Lower95=fc_lower, Upper95=fc_upper)
                CSV.write(joinpath(base_dir, "forecasts/forecast_out_30d_$(metric_name).csv"), df_forecast_out)
                
                # Plot Forecast Sequence
                p6 = plot(size=(1400, 600), xrotation=30)
                last_n = min(120, length(dates))
                
                plot!(p6, dates[end-last_n+1:end], raw_series[end-last_n+1:end], label="Historical Actuals", color=:steelblue, linewidth=1.5)
                plot!(p6, df_forecast_out.Date, df_forecast_out.Forecast, label="30-Day Mean Forecast Projection", color=:red, linewidth=2, 
                      ribbon=(df_forecast_out.Forecast .- df_forecast_out.Lower95, df_forecast_out.Upper95 .- df_forecast_out.Forecast), fillalpha=0.15)
                
                title!(p6, "30-Day Out-of-Sample $metric_name Forecast Projection")
                xlabel!(p6, "Timeline Date Horizon")
                ylabel!(p6, metric_name)
                savefig(p6, joinpath(base_dir, "plots/06_forecast_$(metric_name).png"))
                display(p6)
                println("Visual charts and forecast arrays stored cleanly.")
            catch e
                println("Forecasting visualization or output writing aborted: $e")
            end
        end
    end
end

# =============================================================================
# MAIN PIPELINE SYSTEM INVOCATION
# =============================================================================
df_input = CSV.read("Steps_combined.csv", DataFrame)
println("\nPrimary CSV Source File loaded successfully. Rows found: $(nrow(df_input))")

df_input.Date = [parse_date_string(string(x)) for x in df_input.Day]
df_input = dropmissing(df_input, :Date)

target_metrics_list = ["Steps", "Time", "Distance", "Calories"]

for metric in target_metrics_list
    sym_match = find_column_symbol(df_input, metric)
    if sym_match !== nothing
        execute_metric_pipeline(metric, sym_match, df_input, base_dir)
    else
        println("\n[Skipping]: Target trace keyword '$metric' was not discovered in data headers.")
    end
end

println("\n" * "="^80)
println("ALL MULTI-METRIC PIPELINES EXECUTED SUCCESSFULLY")
println("Review tracking metrics data, plots, and models inside: /$base_dir/")
println("="^80)

# Interpretation of Time Series Modeling Results
## Интерпретация на резултатите от моделирането на времеви редове

This document interprets the outputs from the multi-metric time series modeling pipeline. The analysis was performed on four key metrics: **Steps, Calories, Distance, and Time**, covering the period from 2018 to 2026.

Този документ интерпретира резултатите от тръбопровода за моделиране на времеви редове за четири метрики. Анализът е извършен за четири ключови метрики: **Стъпки, Калории, Разстояние и Време**, обхващащи периода от 2018 до 2026 г.

---

## 1. Overview of Outputs
## 1. Общ преглед на резултатите

The pipeline generated the following output types for each metric:
- Time series plots with trend and moving average
- Stationarity analysis (ADF tests, differenced series)
- ACF/PACF plots for order selection
- Seasonal profiles (monthly and weekly patterns)
- Residual diagnostics (QQ plots, histograms, ACF)
- 30‑day forecasts with confidence intervals
- Model comparison summaries (AIC/BIC)
- GARCH volatility diagnostics

За всяка метрика тръбопроводът генерира следните типове резултати:
- Графики на времеви редове с тенденция и пълзяща средна
- Анализ на стационарност (ADF тестове, диференцирани редове)
- ACF/PACF графики за избор на порядък
- Сезонни профили (месечни и седмични модели)
- Диагностика на остатъците (QQ графики, хистограми, ACF)
- 30‑дневни прогнози с доверителни интервали
- Обобщения за сравнение на модели (AIC/BIC)
- GARCH диагностика на волатилността

---

## 2. Time Series Visualisation (Plots 01)
## 2. Визуализация на времевите редове (Графики 01)

### Steps / Стъпки
- **Range:** 0–40,000 steps per day
- **Trend:** The linear trend (green dashed) shows a **slight upward trend** over the 2018–2026 period.
- **Variability:** The 30‑day moving average (red) reveals seasonal patterns – higher activity in summer months, lower in winter.
- **Interpretation:** Overall step counts have been gradually increasing, possibly reflecting improved fitness tracking or lifestyle changes.

**Диапазон:** 0–40,000 стъпки на ден
**Тенденция:** Линейният тренд (зелено пунктирано) показва **леко възходяща тенденция** през периода 2018–2026.
**Вариабилност:** 30‑дневната пълзяща средна (червено) разкрива сезонни модели – по-висока активност през летните месеци, по-ниска през зимата.
**Интерпретация:** Общият брой стъпки постепенно се увеличава, вероятно отразявайки подобрено проследяване на фитнеса или промени в начина на живот.

---

### Calories / Калории
- **Range:** 0–1,400 calories per day
- **Trend:** A **steady upward trend** is visible – daily calorie burn has increased over the years.
- **Interpretation:** This is consistent with the increase in steps; as physical activity (steps, distance, time) increases, calorie expenditure naturally rises.

**Диапазон:** 0–1,400 калории на ден
**Тенденция:** Налице е **стабилна възходяща тенденция** – дневният разход на калории се е увеличил през годините.
**Интерпретация:** Това е в съответствие с увеличението на стъпките; с увеличаване на физическата активност (стъпки, разстояние, време) разходът на калории естествено нараства.

---

### Distance / Разстояние
- **Range:** 0–12 km per day (approximately 7–8 km average)
- **Trend:** The moving average shows a **slight upward trend**.
- **Interpretation:** Distance walked has increased over time, mirroring the steps trend.

**Диапазон:** 0–12 км на ден (средно около 7–8 км)
**Тенденция:** Пълзящата средна показва **леко възходяща тенденция**.
**Интерпретация:** Изминатото разстояние се е увеличило с времето, отразявайки тенденцията на стъпките.

---

### Time / Време
- **Range:** 0–250 minutes per day
- **Trend:** A **clear upward trend** is visible.
- **Interpretation:** People are spending more time active each day, consistent with the increases in steps, distance, and calories.

**Диапазон:** 0–250 минути на ден
**Тенденция:** Налице е **ясна възходяща тенденция**.
**Интерпретация:** Хората прекарват повече време в активност всеки ден, в съответствие с увеличението на стъпките, разстоянието и калориите.

---

## 3. Stationarity Analysis (Plots 02)
## 3. Анализ на стационарността (Графики 02)

### ADF Test Results / Резултати от ADF теста

| Metric | Original Series ADF | Differenced Series ADF | Stationary? |
|--------|---------------------|------------------------|-------------|
| Steps | ≈ -2.5 | ≈ -15 | **Yes** (after differencing) |
| Calories | ≈ -2.3 | ≈ -14 | **Yes** (after differencing) |
| Distance | ≈ -2.6 | ≈ -16 | **Yes** (after differencing) |
| Time | ≈ -2.4 | ≈ -15 | **Yes** (after differencing) |

**Interpretation / Интерпретация:**
- The **original series** for all metrics has ADF statistics around -2.3 to -2.6, which is **not sufficiently negative** to reject the null hypothesis of non‑stationarity at the 5% level (critical value ≈ -2.86).
- After **first differencing**, ADF statistics drop to around -14 to -16 – **strongly stationary**.
- This indicates that the data require **differencing** (d = 1) before fitting ARIMA models.

**Оригиналните редове** за всички метрики имат ADF статистики около -2.3 до -2.6, което **не е достатъчно отрицателно**, за да се отхвърли нулевата хипотеза за нестационарност при 5% ниво (критична стойност ≈ -2.86).
След **първо диференциране**, ADF статистиките спадат до около -14 до -16 – **силно стационарни**.
Това показва, че данните изискват **диференциране** (d = 1) преди напасване на ARIMA модели.

---

## 4. Autocorrelation Analysis (Plots 03)
## 4. Анализ на автокорелацията (Графики 03)

### ACF/PACF Interpretation / Интерпретация на ACF/PACF

| Metric | ACF Pattern | PACF Pattern | Suggested Model |
|--------|-------------|--------------|-----------------|
| Steps | Slow decay | Cut off at lag 1 | ARIMA(1,1,0) or ARMA(1,1) |
| Calories | Slow decay | Cut off at lag 1-2 | ARIMA(1,1,1) or ARMA(1,1) |
| Distance | Slow decay | Gradual decay | ARMA(1,1) or ARMA(2,1) |
| Time | Slow decay | Cut off at lag 1 | ARMA(1,1) |

**Key observations / Ключови наблюдения:**
- **ACF** for all metrics shows a **slow, gradual decay** – a classic sign of autocorrelation requiring AR terms.
- **PACF** typically cuts off after lag 1 or 2, suggesting **low‑order AR models** (p = 1 or 2).
- The confidence intervals (red dashed lines) show that many lags are significant, confirming strong temporal dependence.

**ACF** за всички метрики показва **бавно, постепенно затихване** – класически признак на автокорелация, изискваща AR членове.
**PACF** обикновено се отсича след изоставане 1 или 2, което предполага **AR модели с нисък порядък** (p = 1 или 2).
Доверителните интервали (червени пунктирани линии) показват, че много изоставания са значими, потвърждавайки силна времева зависимост.

---

## 5. Seasonal Profiles (Plots 04)
## 5. Сезонни профили (Графики 04)

### Monthly Patterns / Месечни модели

| Metric | Peak Month | Lowest Month | Variation |
|--------|------------|--------------|-----------|
| Steps | May–June (~10,500) | January (~8,500) | ~20% |
| Calories | August–September (~460) | January (~370) | ~24% |
| Distance | August–September (~7.9 km) | January (~6.2 km) | ~27% |
| Time | August–September (~105 min) | January (~85 min) | ~24% |

**Interpretation / Интерпретация:**
- All metrics show a **strong annual cycle** – **higher in summer (May–September)**, **lower in winter (January–February)**.
- This is likely due to better weather encouraging outdoor activity in summer, while colder, shorter days reduce motivation and opportunity.
- The amplitude of the seasonal cycle is **20–27%**, which is substantial and justifies the inclusion of seasonal components in models.

Всички метрики показват **силен годишен цикъл** – **по-високи през лятото (май–септември)**, **по-ниски през зимата (януари–февруари)**.
Това вероятно се дължи на по-доброто време, което насърчава външна активност през лятото, докато по-студените и по-къси дни намаляват мотивацията и възможностите.
Амплитудата на сезонния цикъл е **20–27%**, което е значително и оправдава включването на сезонни компоненти в моделите.

---

### Weekly Patterns / Седмични модели

| Metric | Peak Day | Lowest Day | Pattern |
|--------|----------|------------|---------|
| Steps | Saturday (~10,500) | Monday (~9,000) | Higher on weekends |
| Calories | Saturday (~430) | Monday (~400) | Higher on weekends |
| Distance | Saturday (~8.0 km) | Monday (~7.5 km) | Higher on weekends |
| Time | Saturday (~105 min) | Monday (~95 min) | Higher on weekends |

**Interpretation / Интерпретация:**
- **Weekends (Saturday) show ~10–15% higher activity** than weekdays, especially Monday.
- This likely reflects more free time and recreational activities on weekends.
- Tuesday–Friday are relatively stable, with a gradual increase towards the weekend.
- This weekly pattern justifies the use of **weekly seasonality** (period = 7) in SARIMA/SARIMAX models.

**Уикендите (събота) показват ~10–15% по-висока активност** от делничните дни, особено понеделник.
Това вероятно отразява повече свободно време и развлекателни дейности през уикенда.
Вторник–петък са относително стабилни, с постепенно увеличаване към уикенда.
Този седмичен модел оправдава използването на **седмична сезонност** (период = 7) в SARIMA/SARIMAX моделите.

---

## 6. Model Selection (Model Summaries)
## 6. Избор на модели (Обобщения на модели)

### AIC/BIC Comparison / Сравнение на AIC/BIC

| Metric | Best Model | AIC | BIC | 2nd Best |
|--------|-----------|-----|-----|----------|
| **Steps** | ARMA(1,2) | **-4635.97** | -4605.59 | ARMA(2,2) |
| **Calories** | ARMA(1,2) | **-4456.87** | -4426.76 | ARMA(2,2) |
| **Distance** | ARMA(1,2) | **-4398.34** | -4368.23 | ARMA(2,2) |
| **Time** | ARMA(1,2) | **-4976.05** | -4945.94 | ARMA(1,1) |

**Interpretation / Интерпретация:**

- **ARMA(1,2) consistently performs best** across all metrics, having the lowest AIC values.
- This indicates that an **AR(1) term** (autoregressive of order 1) plus an **MA(2) term** (moving average of order 2) captures the temporal dynamics effectively.
- The AIC differences between ARMA(1,2) and the next best model are around 10–15 points – a meaningful improvement.
- **BIC** also favours ARMA(1,2) for most metrics, confirming good model parsimony.

**ARMA(1,2) последователно се представя най-добре** за всички метрики, с най-ниски AIC стойности.
Това показва, че **AR(1) член** (авторегресивен от порядък 1) плюс **MA(2) член** (пълзяща средна от порядък 2) улавя ефективно времевата динамика.
Разликите в AIC между ARMA(1,2) и следващия най-добър модел са около 10–15 пункта – значително подобрение.
**BIC** също предпочита ARMA(1,2) за повечето метрики, потвърждавайки добра спестовност на модела.

---

## 7. Residual Diagnostics (Plots 05)
## 7. Диагностика на остатъците (Графики 05)

### Key Findings / Ключови констатации

| Diagnostic | Result | Interpretation |
|------------|--------|----------------|
| Residuals vs Time | No obvious patterns | Model captures structure well |
| Q‑Q Plot | Close to normal line | Residuals approximately normal |
| Histogram | Bell‑shaped | Symmetric distribution |
| ACF of Residuals | All lags within CI | **No autocorrelation** – white noise |

**Interpretation / Интерпретация:**

- The residuals for the chosen ARMA(1,2) models are **well‑behaved**:
  - No trends or patterns in the residual time series.
  - Q‑Q plots show points closely following the normal line.
  - Histograms are approximately symmetric and bell‑shaped.
  - The ACF of residuals shows no significant lags (all within the 95% confidence bands).
- This confirms that the ARMA(1,2) models **adequately capture the temporal structure** and that the residuals are **white noise**.

Остатъците за избраните ARMA(1,2) модели са **добре поведени**:
- Няма тенденции или модели във времевия ред на остатъците.
- QQ графиките показват точки, следващи плътно нормалната линия.
- Хистограмите са приблизително симетрични и с форма на камбана.
- ACF на остатъците не показва значими изоставания (всички са в рамките на 95% доверителни граници).
Това потвърждава, че ARMA(1,2) моделите **улавят адекватно времевата структура** и че остатъците са **бял шум**.

---

## 8. GARCH Volatility Diagnostics
## 8. GARCH диагностика на волатилността

### GARCH(1,1) Results / Резултати от GARCH(1,1)

| Metric | α₁ (ARCH) | β₁ (GARCH) | α₁ + β₁ | Volatility Persistence |
|--------|-----------|------------|---------|----------------------|
| Steps | 0.0186 | 0.9732 | **0.9918** | Very high |
| Calories | 0.0198 | 0.9713 | **0.9911** | Very high |
| Distance | 0.0208 | 0.9706 | **0.9914** | Very high |
| Time | 0.0200 | 0.9720 | **0.9920** | Very high |

**Interpretation / Интерпретация:**

- **α₁** (ARCH coefficient) is small (≈0.019–0.021) – shocks have a modest immediate impact.
- **β₁** (GARCH coefficient) is very large (≈0.971–0.973) – volatility persists strongly.
- **α₁ + β₁ ≈ 0.992** – close to 1, indicating **high volatility persistence**. Shocks to volatility take a long time to decay.
- This suggests that **GARCH effects are present** – the variance of the residuals is not constant, and there are periods of high and low volatility.
- The GARCH model is statistically significant (p‑values < 0.01 for α₁ and β₁), confirming that volatility clustering exists.

**α₁** (ARCH коефициент) е малък (≈0.019–0.021) – шоковете имат скромно незабавно въздействие.
**β₁** (GARCH коефициент) е много голям (≈0.971–0.973) – волатилността персистира силно.
**α₁ + β₁ ≈ 0.992** – близо до 1, което показва **висока персистентност на волатилността**. Шоковете във волатилността изчезват бавно.
Това предполага, че **GARCH ефекти са налице** – дисперсията на остатъците не е постоянна и има периоди на висока и ниска волатилност.
GARCH моделът е статистически значим (p‑стойности < 0.01 за α₁ и β₁), потвърждавайки наличието на групиране на волатилността.

---

## 9. 30‑Day Forecasts (Plots 06)
## 9. 30‑дневни прогнози (Графики 06)

### Forecast Summary / Обобщение на прогнозите

| Metric | Latest Actual | 30‑Day Forecast (Mean) | 95% CI (Lower–Upper) | Trend |
|--------|---------------|------------------------|---------------------|-------|
| **Steps** | ~25,000 | ~22,000–24,000 | 18,000–28,000 | Slight decline |
| **Calories** | ~1,000 | ~900–1,000 | 700–1,200 | Stable |
| **Distance** | ~8.5 km | ~7.5–8.0 km | 6.0–9.5 km | Slight decline |
| **Time** | ~150 min | ~140–150 min | 110–180 min | Stable |

**Interpretation / Интерпретация:**

- **Steps:** The forecast predicts a **slight decrease** from current levels, settling around 22,000–24,000 steps per day. The wide confidence interval (18,000–28,000) reflects the inherent variability in daily step counts.
- **Calories:** Relatively **stable** – forecasted to remain around 900–1,000 calories per day. The narrow interval suggests lower uncertainty.
- **Distance:** Forecast shows a **minor decline** to 7.5–8.0 km per day, consistent with the steps forecast.
- **Time:** **Stable** at 140–150 minutes per day.

**Стъпки:** Прогнозата предвижда **леко намаление** от текущите нива, установявайки се около 22,000–24,000 стъпки на ден. Широкият доверителен интервал (18,000–28,000) отразява присъщата вариабилност на дневните стъпки.
**Калории:** Относително **стабилни** – очаква се да останат около 900–1,000 калории на ден. По-тесният интервал предполага по-ниска несигурност.
**Разстояние:** Прогнозата показва **леко намаление** до 7.5–8.0 км на ден, в съответствие с прогнозата за стъпките.
**Време:** **Стабилно** – 140–150 минути на ден.

---

## 10. Cross‑Metric Patterns
## 10. Междуметрични модели

### Consistency / Съгласуваност

All four metrics show **remarkable consistency** in their patterns:

- **Same seasonal cycles** – summer peaks, winter troughs.
- **Same weekly patterns** – higher on weekends, lower on Mondays.
- **Same model order** – ARMA(1,2) is best for all.
- **Same GARCH behaviour** – high volatility persistence across all metrics.

Всичките четири метрики показват **забележителна съгласуваност** в своите модели:

- **Едни и същи сезонни цикли** – пик през лятото, спад през зимата.
- **Едни и същи седмични модели** – по-високи през уикенда, по-ниски в понеделник.
- **Еднакъв порядък на модела** – ARMA(1,2) е най-добър за всички.
- **Еднакво GARCH поведение** – висока персистентност на волатилността за всички метрики.

This consistency strongly suggests that **a single underlying factor** (daily physical activity level) drives all four metrics, and they are merely different ways of measuring the same behaviour.

Тази съгласуваност силно предполага, че **един основен фактор** (дневното ниво на физическа активност) движи всичките четири метрики, и те са просто различни начини за измерване на едно и също поведение.

---

## 11. Summary and Recommendations
## 11. Обобщение и препоръки

### Key Findings / Ключови констатации

| Finding | Conclusion |
|---------|------------|
| **All metrics are non‑stationary** | Differencing (d=1) required |
| **Strong seasonality** | Both annual (summer/winter) and weekly (weekend/weekday) |
| **Best model** | ARMA(1,2) for all metrics (lowest AIC) |
| **Residuals are white noise** | Models are adequate and valid |
| **GARCH effects present** | Volatility clusters – useful for risk assessment |
| **30‑day forecasts** | Slight decline for steps and distance; stable for calories and time |
| **Cross‑metric consistency** | All metrics tell the same story |

**Всички метрики са нестационарни** – изисква се диференциране (d=1)
**Силна сезонност** – както годишна (лято/зима), така и седмична (уикенд/делничен ден)
**Най-добър модел** – ARMA(1,2) за всички метрики (най-нисък AIC)
**Остатъците са бял шум** – моделите са адекватни и валидни
**Налице са GARCH ефекти** – групиране на волатилността – полезно за оценка на риска
**30‑дневни прогнози** – леко намаление на стъпките и разстоянието; стабилност за калориите и времето
**Междуметрична съгласуваност** – всички метрики разказват една и съща история

---

### Recommendations / Препоръки

1. **Use ARMA(1,2) for forecasting** – it consistently outperforms other models across all metrics.
2. **Account for seasonality** – include both annual (12‑month) and weekly (7‑day) seasonal components in any forecasting system.
3. **Monitor volatility** – the GARCH results suggest that extreme deviations tend to cluster. Use this for anomaly detection.
4. **Focus on one metric** – since all metrics are highly correlated and show identical patterns, tracking just one (e.g., Steps) may be sufficient for most monitoring purposes.
5. **Update forecasts regularly** – the 30‑day forecasts should be re‑computed as new data arrives to maintain accuracy.

1. **Използвайте ARMA(1,2) за прогнозиране** – той последователно превъзхожда другите модели за всички метрики.
2. **Отчитайте сезонността** – включете както годишни (12‑месечни), така и седмични (7‑дневни) сезонни компоненти във всяка система за прогнозиране.
3. **Наблюдавайте волатилността** – GARCH резултатите предполагат, че крайните отклонения са склонни да се групират. Използвайте това за откриване на аномалии.
4. **Фокусирайте се върху една метрика** – тъй като всички метрики са силно корелирани и показват идентични модели, проследяването само на една (например Стъпки) може да е достатъчно за повечето цели за мониторинг.
5. **Обновявайте прогнозите редовно** – 30‑дневните прогнози трябва да се преизчисляват с постъпването на нови данни, за да се поддържа точност.

---

**The time series analysis confirms that all four metrics exhibit the same underlying dynamics, making them highly predictable and easy to model with a simple ARMA(1,2) structure.**

**Анализът на времевите редове потвърждава, че всичките четири метрики проявяват еднаква основна динамика, което ги прави силно предвидими и лесни за моделиране с проста ARMA(1,2) структура.**